
# CABLE paper-ready experiment — lean 10-run protocol

## Final design

**CABLE = Context-Aware Boundary and Local Evidence**

The CABLE architecture is frozen:

\[
Y = X + \Delta_{\rm local} + \alpha\,\Delta_{\rm context},
\]

with:

- frozen H4-K3 MHTM-Local as the full-resolution local-evidence branch;
- 16×16 coarse global self-attention as the context branch;
- 4 context heads;
- learned context scale \(\alpha\), initialized to 0.1;
- original SwiftMSeg LGL blocks at encoder Stages 2–4;
- ordinary cross-entropy training only.

No architecture or loss tuning is performed in this notebook.

---

## Why this notebook trains only CABLE

The authoritative SwiftMSeg-LGL and MHTM-Local experiments are already complete:

- **BUS-BRA:** use the clean checkpoint-only audit generated from the saved best-validation-Dice checkpoints;
- **BUS-UCLM:** use the completed supervised patient-disjoint five-fold experiment.

Those two baselines therefore remain exactly as previously frozen.

This notebook performs only:

> **5 BUS-BRA CABLE trainings + 5 BUS-UCLM CABLE trainings = 10 new trainings.**

For every CABLE fold, the same 50-epoch trajectory saves **two validation-selected checkpoints**:

1. **CABLE-Dice** — maximum validation foreground Dice.  
   This is the fair architecture-comparison operating point because Swift-LGL and MHTM
   were also frozen by validation Dice.

2. **CABLE-Boundary** — among epochs within **0.005 absolute Dice** of the maximum,
   select the minimum validation HD95; ties are resolved by higher Boundary-IoU,
   then higher Dice, then earlier epoch.

The held-out outer test fold is accessed only after both checkpoints have been frozen.

### Paper-facing interpretation

- **Main architecture table:** Swift-LGL vs MHTM-Local vs **CABLE-Dice**.
- **Secondary ultrasound operating-point analysis:** **CABLE-Dice vs CABLE-Boundary**.

This prevents CABLE from receiving an unfair checkpoint-selection advantage while still
testing the medically motivated boundary-aware operating point.



## Dataset protocol

### BUS-BRA
The dataset-supplied official 5-fold partition is the outer patient-disjoint test split.
For each outer fold, the inner validation split is reproduced from the remaining patients
using the same frozen seed as the previous final experiment.

### BUS-UCLM
The same geometry audit is retained: 13 incompatible HESN image-mask pairs are excluded,
leaving 670 aligned images from 37 patients. The **exact previously saved outer-fold
assignment** is loaded from the completed BUS-UCLM experiment rather than regenerated.

The primary supervised task remains lesion segmentation. Normal images are excluded from
training and checkpoint selection and are evaluated only as a held-out false-positive
control.


In [1]:

# ============================================================
# 0. INSTALL + IMPORTS
# ============================================================

import sys
import subprocess
import importlib.util

packages = {
    "transformers": "transformers>=4.45",
    "albumentations": "albumentations>=1.4",
    "cv2": "opencv-python-headless",
    "sklearn": "scikit-learn",
    "requests": "requests",
    "tqdm": "tqdm",
}

for module, package in packages.items():
    if importlib.util.find_spec(module) is None:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", package]
        )

import os
import gc
import json
import time
import math
import random
import zipfile
import warnings
import hashlib
import shutil
import datetime
import tempfile

from dataclasses import dataclass, asdict
from pathlib import Path

import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt

from scipy.ndimage import binary_erosion, distance_transform_edt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import StratifiedGroupKFold

import albumentations as A
from albumentations.pytorch import ToTensorV2

from transformers import SegformerModel
from IPython.display import display

warnings.filterwarnings("ignore")

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("PyTorch:", torch.__version__)
print("Device :", DEVICE)

if DEVICE.type == "cuda":
    print("GPU    :", torch.cuda.get_device_name(0))


PyTorch: 2.11.0+cu128
Device : cuda
GPU    : Tesla T4


In [36]:

# ============================================================
# 1. FINAL FROZEN CONFIGURATION
# ============================================================

from dataclasses import dataclass, asdict

@dataclass
class CFG:
    image_size: int = 256
    batch_size: int = 8
    eval_batch_size: int = 8

    busbra_num_workers: int = 4
    uclm_num_workers: int = 2

    num_classes: int = 2

    epochs: int = 50
    lr: float = 1e-3
    weight_decay: float = 1e-2
    scheduler_factor: float = 0.5
    scheduler_patience: int = 3

    pretrained_backbone: str = "nvidia/mit-b0"
    lgl_heads: int = 4

    mhtm_heads: int = 4
    mhtm_kernel: int = 3

    cable_context_hw: int = 16
    cable_context_heads: int = 4
    cable_context_scale_init: float = 0.1

    train_seed: int = 42

    busbra_inner_val_seed: int = 2026
    uclm_inner_val_seed: int = 3026

    use_amp: bool = False

    # Boundary-aware operating point:
    dice_tolerance: float = 0.005
    hd95_tie_tolerance: float = 1e-12
    boundary_iou_tie_tolerance: float = 1e-12


cfg = CFG()

BASELINE_VARIANTS = (
    "swift_lgl",
    "mhtm_hybrid_final",
)

CABLE_VARIANT = "cable_s1"

ARCHITECTURE_VARIANTS = (
    "swift_lgl",
    "mhtm_hybrid_final",
    "cable_s1",
)

OUTER_FOLDS = (
    1,
    2,
    3,
    4,
    5,
)

RUN_BUSBRA = True
RUN_BUSUCLM = True

FORCE_RERUN = False

print(json.dumps(asdict(cfg), indent=2))
print("Baseline variants reused :", BASELINE_VARIANTS)
print("Only trained variant     :", CABLE_VARIANT)
print("Outer folds              :", OUTER_FOLDS)
print("Total new trainings      :", len(OUTER_FOLDS) * 2)


{
  "image_size": 256,
  "batch_size": 8,
  "eval_batch_size": 8,
  "busbra_num_workers": 4,
  "uclm_num_workers": 2,
  "num_classes": 2,
  "epochs": 50,
  "lr": 0.001,
  "weight_decay": 0.01,
  "scheduler_factor": 0.5,
  "scheduler_patience": 3,
  "pretrained_backbone": "nvidia/mit-b0",
  "lgl_heads": 4,
  "mhtm_heads": 4,
  "mhtm_kernel": 3,
  "cable_context_hw": 16,
  "cable_context_heads": 4,
  "cable_context_scale_init": 0.1,
  "train_seed": 42,
  "busbra_inner_val_seed": 2026,
  "uclm_inner_val_seed": 3026,
  "use_amp": false,
  "dice_tolerance": 0.005,
  "hd95_tie_tolerance": 1e-12,
  "boundary_iou_tie_tolerance": 1e-12
}
Baseline variants reused : ('swift_lgl', 'mhtm_hybrid_final')
Only trained variant     : cable_s1
Outer folds              : (1, 2, 3, 4, 5)
Total new trainings      : 10


In [37]:

# ============================================================
# 2. REPRODUCIBILITY
# ============================================================

def set_seed(seed: int):
    os.environ["PYTHONHASHSEED"] = str(seed)

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    try:
        torch.use_deterministic_algorithms(
            True,
            warn_only=True,
        )
    except Exception:
        pass


def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % (2**32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)


set_seed(cfg.train_seed)


In [38]:

# ============================================================
# 3. GOOGLE DRIVE + AUTHORITATIVE ROOTS
# ============================================================

IN_COLAB = False

try:
    import google.colab
    IN_COLAB = True
except Exception:
    pass


if IN_COLAB:
    from google.colab import drive

    drive.mount(
        "/content/drive",
        force_remount=False,
    )

    DRIVE_ROOT = Path(
        "/content/drive/MyDrive/MHTM_MEDICAL"
    )

    LOCAL_ROOT = Path(
        "/content/MHTM_MEDICAL"
    )

else:
    DRIVE_ROOT = Path("./MHTM_MEDICAL")
    LOCAL_ROOT = Path("./MHTM_MEDICAL_LOCAL")


DRIVE_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

LOCAL_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


# ------------------------------------------------------------
# AUTHORITATIVE EXISTING BASELINES — NEVER MODIFIED HERE
# ------------------------------------------------------------

# BUS-BRA authoritative metrics after the clean checkpoint audit.
OLD_BRA_AUDIT_ROOT = (
    DRIVE_ROOT
    / "SWIFT_MHTM_FINAL_OFFICIALCV_CLEAN_AUDIT_V1"
)

# Original BUS-BRA training root is kept only for checkpoint provenance /
# optional future qualitative regeneration. It is NOT retrained here.
OLD_BRA_TRAIN_ROOT = (
    DRIVE_ROOT
    / "SWIFT_MHTM_FINAL_OFFICIALCV_V1"
)

# BUS-UCLM completed supervised CV root.
OLD_UCLM_ROOT = (
    DRIVE_ROOT
    / "SWIFT_MHTM_BUSUCLM_SUPERVISED_CV_V1"
)


# ------------------------------------------------------------
# NEW CABLE-ONLY OUTPUT
# ------------------------------------------------------------

FINAL_ROOT = (
    DRIVE_ROOT
    / "CABLE_PAPER_READY_LEAN10_V1"
)

FINAL_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

BUSBRA_FINAL_ROOT = (
    FINAL_ROOT
    / "BUS_BRA"
)

UCLM_FINAL_ROOT = (
    FINAL_ROOT
    / "BUS_UCLM"
)

BUSBRA_FINAL_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

UCLM_FINAL_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


print("BUS-BRA clean baseline root :", OLD_BRA_AUDIT_ROOT)
print("BUS-UCLM baseline root      :", OLD_UCLM_ROOT)
print("New CABLE output root       :", FINAL_ROOT)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
BUS-BRA clean baseline root : /content/drive/MyDrive/MHTM_MEDICAL/SWIFT_MHTM_FINAL_OFFICIALCV_CLEAN_AUDIT_V1
BUS-UCLM baseline root      : /content/drive/MyDrive/MHTM_MEDICAL/SWIFT_MHTM_BUSUCLM_SUPERVISED_CV_V1
New CABLE output root       : /content/drive/MyDrive/MHTM_MEDICAL/CABLE_PAPER_READY_LEAN10_V1


In [39]:

# ============================================================
# 4. SHARED MODEL: SWIFT-LGL / MHTM / CABLE
# ============================================================

class LGLBlock(nn.Module):
    def __init__(
        self,
        dim,
        num_heads=4,
    ):
        super().__init__()

        self.local1 = nn.Conv2d(
            dim,
            dim,
            kernel_size=3,
            padding=1,
            groups=dim,
        )

        self.norm1 = nn.LayerNorm(
            dim
        )

        self.attn = nn.MultiheadAttention(
            embed_dim=dim,
            num_heads=num_heads,
            batch_first=True,
        )

        self.norm2 = nn.LayerNorm(
            dim
        )

        self.local2 = nn.Conv2d(
            dim,
            dim,
            kernel_size=3,
            padding=1,
            groups=dim,
        )

    def forward(
        self,
        x,
    ):
        B, C, H, W = x.shape

        x_local = self.local1(
            x
        )

        x_flat = (
            x_local
            .permute(0, 2, 3, 1)
            .reshape(B, H * W, C)
        )

        x_norm = self.norm1(
            x_flat
        )

        attn_out, _ = self.attn(
            x_norm,
            x_norm,
            x_norm,
            need_weights=False,
        )

        x_attn = self.norm2(
            x_norm
            + attn_out
        )

        x_attn = (
            x_attn
            .reshape(B, H, W, C)
            .permute(0, 3, 1, 2)
        )

        return self.local2(
            x_attn
        )


class CanonicalMHTMLocal2D(nn.Module):
    def __init__(
        self,
        channels,
        num_heads=4,
        kernel_size=3,
    ):
        super().__init__()

        if channels % num_heads != 0:
            raise ValueError(
                "channels must be divisible by num_heads"
            )

        self.local_mix = nn.Conv2d(
            channels,
            channels,
            kernel_size=kernel_size,
            padding=kernel_size // 2,
            groups=channels,
            bias=False,
        )

        self.head_proj = nn.Conv2d(
            channels,
            channels,
            kernel_size=1,
            groups=num_heads,
            bias=False,
        )

        self.out_proj = nn.Conv2d(
            channels,
            channels,
            kernel_size=1,
            bias=False,
        )

    def forward(
        self,
        x,
    ):
        y = self.local_mix(
            x
        )

        y = self.head_proj(
            y
        )

        y = self.out_proj(
            y
        )

        return x + y


class CABLEBlock(nn.Module):
    """
    Context-Aware Boundary and Local Evidence.

    Fusion:
        x + local_delta + alpha * coarse_context_delta
    """

    def __init__(
        self,
        channels,
        local_heads=4,
        local_kernel=3,
        context_hw=16,
        context_heads=4,
        context_scale_init=0.1,
    ):
        super().__init__()

        if channels % local_heads != 0:
            raise ValueError(
                "channels must be divisible by local_heads"
            )

        if channels % context_heads != 0:
            raise ValueError(
                "channels must be divisible by context_heads"
            )

        self.channels = int(
            channels
        )

        self.context_hw = int(
            context_hw
        )

        self.local_mix = nn.Conv2d(
            channels,
            channels,
            kernel_size=local_kernel,
            padding=local_kernel // 2,
            groups=channels,
            bias=False,
        )

        self.local_head_proj = nn.Conv2d(
            channels,
            channels,
            kernel_size=1,
            groups=local_heads,
            bias=False,
        )

        self.local_out_proj = nn.Conv2d(
            channels,
            channels,
            kernel_size=1,
            bias=False,
        )

        self.context_norm1 = nn.LayerNorm(
            channels
        )

        self.context_attn = nn.MultiheadAttention(
            embed_dim=channels,
            num_heads=context_heads,
            batch_first=True,
        )

        self.context_norm2 = nn.LayerNorm(
            channels
        )

        self.context_proj = nn.Conv2d(
            channels,
            channels,
            kernel_size=1,
            bias=False,
        )

        self.context_scale = nn.Parameter(
            torch.tensor(
                float(
                    context_scale_init
                )
            )
        )

    def forward(
        self,
        x,
    ):
        B, C, H, W = x.shape

        local_delta = self.local_mix(
            x
        )

        local_delta = self.local_head_proj(
            local_delta
        )

        local_delta = self.local_out_proj(
            local_delta
        )

        ctx = F.adaptive_avg_pool2d(
            x,
            output_size=(
                self.context_hw,
                self.context_hw,
            ),
        )

        h = self.context_hw
        w = self.context_hw

        tokens = (
            ctx
            .permute(0, 2, 3, 1)
            .reshape(B, h * w, C)
        )

        tokens_norm = self.context_norm1(
            tokens
        )

        attn_out, _ = self.context_attn(
            tokens_norm,
            tokens_norm,
            tokens_norm,
            need_weights=False,
        )

        tokens = self.context_norm2(
            tokens_norm
            + attn_out
        )

        ctx = (
            tokens
            .reshape(B, h, w, C)
            .permute(0, 3, 1, 2)
        )

        ctx = self.context_proj(
            ctx
        )

        context_delta = F.interpolate(
            ctx,
            size=(H, W),
            mode="bilinear",
            align_corners=False,
        )

        return (
            x
            + local_delta
            + self.context_scale
            * context_delta
        )


def make_aggregator(
    name,
    channels,
):
    if name == "swift_lgl":
        return LGLBlock(
            channels,
            num_heads=cfg.lgl_heads,
        )

    if name == "mhtm_local":
        return CanonicalMHTMLocal2D(
            channels,
            num_heads=cfg.mhtm_heads,
            kernel_size=cfg.mhtm_kernel,
        )

    if name == "cable":
        return CABLEBlock(
            channels,
            local_heads=cfg.mhtm_heads,
            local_kernel=cfg.mhtm_kernel,
            context_hw=cfg.cable_context_hw,
            context_heads=cfg.cable_context_heads,
            context_scale_init=cfg.cable_context_scale_init,
        )

    raise ValueError(
        name
    )


class SwiftMSegFamily(nn.Module):
    CHANNELS = (
        32,
        64,
        160,
        256,
    )

    def __init__(
        self,
        variant,
        num_classes=2,
    ):
        super().__init__()

        self.backbone = (
            SegformerModel
            .from_pretrained(
                cfg.pretrained_backbone,
                output_hidden_states=True,
            )
        )

        # Shared decoder constructed before variant-specific modules.
        self.up3 = nn.ConvTranspose2d(
            256,
            160,
            2,
            stride=2,
        )

        self.dec3 = nn.Conv2d(
            160 + 160,
            160,
            3,
            padding=1,
        )

        self.up2 = nn.ConvTranspose2d(
            160,
            64,
            2,
            stride=2,
        )

        self.dec2 = nn.Conv2d(
            64 + 64,
            64,
            3,
            padding=1,
        )

        self.up1 = nn.ConvTranspose2d(
            64,
            32,
            2,
            stride=2,
        )

        self.dec1 = nn.Conv2d(
            32 + 32,
            32,
            3,
            padding=1,
        )

        self.final = nn.Conv2d(
            32,
            num_classes,
            kernel_size=1,
        )

        if variant == "swift_lgl":
            stage_names = (
                "swift_lgl",
                "swift_lgl",
                "swift_lgl",
                "swift_lgl",
            )

        elif variant == "mhtm_hybrid_final":
            stage_names = (
                "mhtm_local",
                "swift_lgl",
                "swift_lgl",
                "swift_lgl",
            )

        elif variant == "cable_s1":
            stage_names = (
                "cable",
                "swift_lgl",
                "swift_lgl",
                "swift_lgl",
            )

        else:
            raise ValueError(
                variant
            )

        self.aggregators = nn.ModuleList([
            make_aggregator(
                name,
                channels,
            )
            for name, channels
            in zip(
                stage_names,
                self.CHANNELS,
            )
        ])

        self.variant = variant
        self.stage_names = stage_names

    def forward(
        self,
        x,
    ):
        outputs = self.backbone(
            pixel_values=x,
            output_hidden_states=True,
            return_dict=True,
        )

        feats = outputs.hidden_states

        if len(feats) != 4:
            raise RuntimeError(
                f"Expected 4 encoder stages, got {len(feats)}"
            )

        e1, e2, e3, e4 = [
            agg(feat)
            for agg, feat
            in zip(
                self.aggregators,
                feats,
            )
        ]

        d3 = self.up3(
            e4
        )

        d3 = torch.cat(
            [d3, e3],
            dim=1,
        )

        d3 = F.relu(
            self.dec3(d3),
            inplace=False,
        )

        d2 = self.up2(
            d3
        )

        d2 = torch.cat(
            [d2, e2],
            dim=1,
        )

        d2 = F.relu(
            self.dec2(d2),
            inplace=False,
        )

        d1 = self.up1(
            d2
        )

        d1 = torch.cat(
            [d1, e1],
            dim=1,
        )

        d1 = F.relu(
            self.dec1(d1),
            inplace=False,
        )

        out = self.final(
            d1
        )

        return F.interpolate(
            out,
            size=x.shape[-2:],
            mode="bilinear",
            align_corners=False,
        )


In [40]:

# ============================================================
# 5. MODEL / COMPLEXITY AUDIT
# ============================================================

def count_trainable_params(
    model,
):
    return sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )


def aggregator_params(
    model,
):
    return sum(
        p.numel()
        for p in model.aggregators.parameters()
        if p.requires_grad
    )


STAGES = [
    (64, 64, 32),
    (32, 32, 64),
    (16, 16, 160),
    (8, 8, 256),
]


def lgl_macs(
    H,
    W,
    C,
    k=3,
):
    N = H * W

    return (
        2 * N * C * k * k
        + 4 * N * C * C
        + 2 * N * N * C
    )


def mhtm_macs(
    H,
    W,
    C,
    k=3,
    heads=4,
):
    N = H * W

    return (
        N * C * k * k
        + N * C * C / heads
        + N * C * C
    )


def coarse_context_macs(
    context_hw,
    C,
):
    N = context_hw * context_hw

    # QKV + output projection + context_proj + QK + AV.
    return (
        5 * N * C * C
        + 2 * N * N * C
    )


lgl_costs = [
    lgl_macs(
        *stage
    )
    for stage
    in STAGES
]

swift_cost = sum(
    lgl_costs
)

mhtm_cost = (
    mhtm_macs(
        *STAGES[0],
        k=cfg.mhtm_kernel,
        heads=cfg.mhtm_heads,
    )
    + sum(
        lgl_costs[1:]
    )
)

cable_cost = (
    mhtm_cost
    + coarse_context_macs(
        cfg.cable_context_hw,
        STAGES[0][2],
    )
)


complexity_lookup = {
    "swift_lgl": swift_cost / 1e9,
    "mhtm_hybrid_final": mhtm_cost / 1e9,
    "cable_s1": cable_cost / 1e9,
}


audit_rows = []
shared_reference = None

for variant in ARCHITECTURE_VARIANTS:
    set_seed(
        777
    )

    model = SwiftMSegFamily(
        variant=variant,
        num_classes=cfg.num_classes,
    ).to(
        DEVICE
    )

    shared = {
        k: v.detach().cpu()
        for k, v
        in model.state_dict().items()
        if not k.startswith(
            "aggregators."
        )
    }

    if shared_reference is None:
        shared_reference = {
            k: v.clone()
            for k, v
            in shared.items()
        }

    shared_exact = all(
        torch.equal(
            shared_reference[k],
            shared[k],
        )
        for k
        in shared_reference
    )

    dummy = torch.zeros(
        1,
        3,
        cfg.image_size,
        cfg.image_size,
        device=DEVICE,
    )

    with torch.inference_mode():
        output = model(
            dummy
        )

    audit_rows.append({
        "variant": variant,
        "params_M": count_trainable_params(
            model
        ) / 1e6,
        "aggregator_params_M": aggregator_params(
            model
        ) / 1e6,
        "aggregator_GMACs": complexity_lookup[
            variant
        ],
        "shared_initialization_exact": shared_exact,
        "output_shape": tuple(
            output.shape
        ),
    })

    del model
    gc.collect()

    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()


architecture_audit = pd.DataFrame(
    audit_rows
)

display(
    architecture_audit
)

assert architecture_audit[
    "shared_initialization_exact"
].all()

print("CABLE aggregation reduction vs Swift-LGL:",
      1.0 - cable_cost / swift_cost)


Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

[transformers] SegformerModel LOAD REPORT from: nvidia/mit-b0
Key               | Status     |  | 
------------------+------------+--+-
classifier.weight | UNEXPECTED |  | 
classifier.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

[transformers] SegformerModel LOAD REPORT from: nvidia/mit-b0
Key               | Status     |  | 
------------------+------------+--+-
classifier.weight | UNEXPECTED |  | 
classifier.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

[transformers] SegformerModel LOAD REPORT from: nvidia/mit-b0
Key               | Status     |  | 
------------------+------------+--+-
classifier.weight | UNEXPECTED |  | 
classifier.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


,variant,params_M,aggregator_params_M,aggregator_GMACs,shared_initialization_exact,output_shape
0,swift_lgl,4.485282,0.399360,1.312145,True,"(1, 2, 256, 256)"
1,mhtm_hybrid_final,4.481858,0.395936,0.225690,True,"(1, 2, 256, 256)"
2,cable_s1,4.487235,0.401313,0.231195,True,"(1, 2, 256, 256)"


CABLE aggregation reduction vs Swift-LGL: 0.823804113027083


In [41]:

# ============================================================
# 6. SHARED SEGMENTATION METRICS
# ============================================================

criterion = nn.CrossEntropyLoss()


def confusion_counts(
    pred_fg,
    target_fg,
):
    pred_fg = np.asarray(
        pred_fg,
        dtype=bool,
    )

    target_fg = np.asarray(
        target_fg,
        dtype=bool,
    )

    tp = int(
        np.logical_and(
            pred_fg,
            target_fg,
        ).sum()
    )

    tn = int(
        np.logical_and(
            ~pred_fg,
            ~target_fg,
        ).sum()
    )

    fp = int(
        np.logical_and(
            pred_fg,
            ~target_fg,
        ).sum()
    )

    fn = int(
        np.logical_and(
            ~pred_fg,
            target_fg,
        ).sum()
    )

    return tp, tn, fp, fn


def dice_iou(
    tp,
    fp,
    fn,
):
    union_den = tp + fp + fn
    dice_den = 2 * tp + fp + fn

    if union_den == 0:
        return 1.0, 1.0

    return (
        float(
            2 * tp / dice_den
        ),
        float(
            tp / union_den
        ),
    )


def hd95_binary(
    pred_fg,
    target_fg,
):
    pred_fg = np.asarray(
        pred_fg,
        dtype=bool,
    )

    target_fg = np.asarray(
        target_fg,
        dtype=bool,
    )

    if (
        not pred_fg.any()
        and not target_fg.any()
    ):
        return 0.0

    if (
        pred_fg.any()
        != target_fg.any()
    ):
        H, W = pred_fg.shape

        return float(
            np.sqrt(
                H * H
                + W * W
            )
        )

    pred_boundary = np.logical_xor(
        pred_fg,
        binary_erosion(
            pred_fg
        ),
    )

    target_boundary = np.logical_xor(
        target_fg,
        binary_erosion(
            target_fg
        ),
    )

    if not pred_boundary.any():
        pred_boundary = pred_fg

    if not target_boundary.any():
        target_boundary = target_fg

    dist_to_target = distance_transform_edt(
        ~target_boundary
    )

    dist_to_pred = distance_transform_edt(
        ~pred_boundary
    )

    distances = np.concatenate([
        dist_to_target[
            pred_boundary
        ],
        dist_to_pred[
            target_boundary
        ],
    ])

    return float(
        np.percentile(
            distances,
            95,
        )
    )


def boundary_iou(
    pred_fg,
    target_fg,
    width=3,
):
    pred_fg = np.asarray(
        pred_fg,
        dtype=bool,
    )

    target_fg = np.asarray(
        target_fg,
        dtype=bool,
    )

    structure = np.ones(
        (
            2 * width + 1,
            2 * width + 1,
        ),
        dtype=bool,
    )

    pred_eroded = binary_erosion(
        pred_fg,
        structure=structure,
        border_value=0,
    )

    target_eroded = binary_erosion(
        target_fg,
        structure=structure,
        border_value=0,
    )

    pred_b = np.logical_and(
        pred_fg,
        ~pred_eroded,
    )

    target_b = np.logical_and(
        target_fg,
        ~target_eroded,
    )

    inter = int(
        np.logical_and(
            pred_b,
            target_b,
        ).sum()
    )

    union = int(
        np.logical_or(
            pred_b,
            target_b,
        ).sum()
    )

    if union == 0:
        return 1.0

    return float(
        inter / union
    )


def summarize_lesion_rows(
    details,
):
    expected_dice = (
        2
        * details["iou"].to_numpy()
        / (
            1
            + details["iou"].to_numpy()
        )
    )

    identity_error = float(
        np.max(
            np.abs(
                details["dice"].to_numpy()
                - expected_dice
            )
        )
    )

    assert identity_error < 1e-10

    return {
        "dice": float(
            details["dice"].mean()
        ),
        "iou": float(
            details["iou"].mean()
        ),
        "hd95": float(
            details["hd95"].mean()
        ),
        "boundary_iou": float(
            details["boundary_iou"].mean()
        ),
        "n_images": int(
            len(details)
        ),
        "n_patients": int(
            details["patient"].nunique()
        ),
        "dice_iou_identity_error": identity_error,
    }


@torch.inference_mode()
def evaluate_lesion_loader(
    model,
    loader,
    save_masks_path=None,
):
    model.eval()

    losses = []
    rows = []
    pred_masks = []
    pred_ids = []

    for images, masks, meta in loader:
        images = images.to(
            DEVICE,
            non_blocking=True,
        )

        masks_gpu = masks.to(
            DEVICE,
            non_blocking=True,
        )

        logits = model(
            images
        )

        loss = criterion(
            logits,
            masks_gpu,
        )

        losses.append(
            loss.item()
        )

        preds = (
            logits.argmax(dim=1)
            .cpu()
            .numpy()
            .astype(np.uint8)
        )

        targets = (
            masks.numpy()
            .astype(np.uint8)
        )

        for i in range(
            len(preds)
        ):
            p = preds[i] == 1
            t = targets[i] == 1

            tp, tn, fp, fn = confusion_counts(
                p,
                t,
            )

            d, j = dice_iou(
                tp,
                fp,
                fn,
            )

            rows.append({
                "id": str(
                    meta["id"][i]
                ),
                "patient": str(
                    meta["patient"][i]
                ),
                "pathology": str(
                    meta["pathology"][i]
                ),
                "tp": tp,
                "tn": tn,
                "fp": fp,
                "fn": fn,
                "dice": d,
                "iou": j,
                "hd95": hd95_binary(
                    p,
                    t,
                ),
                "boundary_iou": boundary_iou(
                    p,
                    t,
                ),
                "gt_area_fraction": float(
                    t.mean()
                ),
                "pred_area_fraction": float(
                    p.mean()
                ),
            })

            if save_masks_path is not None:
                pred_masks.append(
                    preds[i]
                )
                pred_ids.append(
                    str(
                        meta["id"][i]
                    )
                )

    details = pd.DataFrame(
        rows
    )

    summary = summarize_lesion_rows(
        details
    )

    summary["loss"] = float(
        np.mean(
            losses
        )
    )

    if save_masks_path is not None:
        save_masks_path.parent.mkdir(
            parents=True,
            exist_ok=True,
        )

        np.savez_compressed(
            save_masks_path,
            ids=np.asarray(
                pred_ids,
                dtype=str,
            ),
            masks=np.stack(
                pred_masks,
                axis=0,
            ),
        )

    return summary, details


@torch.inference_mode()
def evaluate_normal_loader(
    model,
    loader,
    save_masks_path=None,
):
    model.eval()

    rows = []
    pred_masks = []
    pred_ids = []

    for images, masks, meta in loader:
        assert int(
            masks.sum()
        ) == 0

        images = images.to(
            DEVICE,
            non_blocking=True,
        )

        logits = model(
            images
        )

        preds = (
            logits.argmax(dim=1)
            .cpu()
            .numpy()
            .astype(np.uint8)
        )

        for i in range(
            len(preds)
        ):
            p = preds[i] == 1

            rows.append({
                "id": str(
                    meta["id"][i]
                ),
                "patient": str(
                    meta["patient"][i]
                ),
                "pathology": "normal",
                "pred_area_fraction": float(
                    p.mean()
                ),
                "any_predicted_lesion": int(
                    p.any()
                ),
            })

            if save_masks_path is not None:
                pred_masks.append(
                    preds[i]
                )
                pred_ids.append(
                    str(
                        meta["id"][i]
                    )
                )

    details = pd.DataFrame(
        rows
    )

    summary = {
        "normal_fp_area_mean": float(
            details[
                "pred_area_fraction"
            ].mean()
        ),
        "normal_fp_area_median": float(
            details[
                "pred_area_fraction"
            ].median()
        ),
        "normal_any_lesion_rate": float(
            details[
                "any_predicted_lesion"
            ].mean()
        ),
        "n_normal_images": int(
            len(details)
        ),
        "n_normal_patients": int(
            details[
                "patient"
            ].nunique()
        ),
    }

    if save_masks_path is not None:
        save_masks_path.parent.mkdir(
            parents=True,
            exist_ok=True,
        )

        np.savez_compressed(
            save_masks_path,
            ids=np.asarray(
                pred_ids,
                dtype=str,
            ),
            masks=np.stack(
                pred_masks,
                axis=0,
            ),
        )

    return summary, details



## Two checkpoint policies from one training trajectory

For CABLE, the ordinary architecture-comparison checkpoint is

\[
e_{\rm Dice}=\arg\max_e D_e.
\]

The boundary-aware operating point first defines the Dice-preserving candidate set

\[
\mathcal C=
\{e : D_e\ge D_{\max}-0.005\},
\]

then selects

\[
e_{\rm Boundary}=
\arg\min_{e\in\mathcal C} H_{95,e}.
\]

Boundary-IoU, Dice, and earlier epoch are deterministic tie-breakers.

**Neither rule uses the test set.**


In [42]:

# ============================================================
# 7. FINAL DICE-CONSTRAINED BOUNDARY SELECTOR
# ============================================================

def select_boundary_checkpoint(
    history_df,
):
    required = {
        "epoch",
        "val_dice",
        "val_hd95",
        "val_boundary_iou",
    }

    missing = required - set(
        history_df.columns
    )

    assert not missing, (
        f"History missing selector columns: {missing}"
    )

    max_dice = float(
        history_df[
            "val_dice"
        ].max()
    )

    threshold = (
        max_dice
        - cfg.dice_tolerance
    )

    candidates = (
        history_df[
            history_df[
                "val_dice"
            ]
            >= threshold
        ]
        .copy()
        .reset_index(drop=True)
    )

    assert len(
        candidates
    ) > 0

    # Primary: minimum HD95.
    best_hd95 = float(
        candidates[
            "val_hd95"
        ].min()
    )

    candidates = candidates[
        candidates[
            "val_hd95"
        ]
        <= (
            best_hd95
            + cfg.hd95_tie_tolerance
        )
    ].copy()

    # Tie-break 1: maximum Boundary IoU.
    best_biou = float(
        candidates[
            "val_boundary_iou"
        ].max()
    )

    candidates = candidates[
        candidates[
            "val_boundary_iou"
        ]
        >= (
            best_biou
            - cfg.boundary_iou_tie_tolerance
        )
    ].copy()

    # Tie-break 2: higher Dice.
    best_dice = float(
        candidates[
            "val_dice"
        ].max()
    )

    candidates = candidates[
        candidates[
            "val_dice"
        ]
        >= (
            best_dice
            - 1e-12
        )
    ].copy()

    # Tie-break 3: earlier epoch.
    selected = (
        candidates
        .sort_values(
            "epoch",
            ascending=True,
        )
        .iloc[
            0
        ]
    )

    return {
        "max_val_dice": max_dice,
        "dice_threshold": threshold,
        "selected_epoch": int(
            selected[
                "epoch"
            ]
        ),
        "selected_val_dice": float(
            selected[
                "val_dice"
            ]
        ),
        "selected_val_iou": float(
            selected[
                "val_iou"
            ]
        ),
        "selected_val_hd95": float(
            selected[
                "val_hd95"
            ]
        ),
        "selected_val_boundary_iou": float(
            selected[
                "val_boundary_iou"
            ]
        ),
        "candidate_count": int(
            (
                history_df[
                    "val_dice"
                ]
                >= threshold
            ).sum()
        ),
    }


# Synthetic selector sanity check:
_selector_test = pd.DataFrame({
    "epoch": [1, 2, 3],
    "val_dice": [0.8479, 0.8472, 0.8420],
    "val_iou": [0.76, 0.76, 0.75],
    "val_hd95": [18.39, 17.75, 17.00],
    "val_boundary_iou": [0.3143, 0.3140, 0.3130],
})

_selector_out = select_boundary_checkpoint(
    _selector_test
)

assert _selector_out[
    "selected_epoch"
] == 2

print(
    "PASS: selector prefers the lower-HD95 checkpoint "
    "when Dice remains within 0.005."
)


PASS: selector prefers the lower-HD95 checkpoint when Dice remains within 0.005.


In [43]:

# ============================================================
# 8. CABLE TRAINING ENGINE — SAVE BOTH CHECKPOINTS
# ============================================================

criterion = nn.CrossEntropyLoss()


def build_optimizer(
    model,
):
    return torch.optim.AdamW(
        model.parameters(),
        lr=cfg.lr,
        weight_decay=cfg.weight_decay,
    )


def build_scheduler(
    optimizer,
):
    return (
        torch.optim
        .lr_scheduler
        .ReduceLROnPlateau(
            optimizer,
            mode="min",
            factor=cfg.scheduler_factor,
            patience=cfg.scheduler_patience,
        )
    )


def train_one_epoch(
    model,
    loader,
    optimizer,
    scaler,
):
    model.train()

    losses = []

    for batch_idx, (
        images,
        masks,
        _,
    ) in enumerate(
        loader
    ):
        images = images.to(
            DEVICE,
            non_blocking=True,
        )

        masks = masks.to(
            DEVICE,
            non_blocking=True,
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        with torch.cuda.amp.autocast(
            enabled=(
                cfg.use_amp
                and DEVICE.type
                == "cuda"
            )
        ):
            logits = model(
                images
            )

            loss = criterion(
                logits,
                masks,
            )

        scaler.scale(
            loss
        ).backward()

        scaler.step(
            optimizer
        )

        scaler.update()

        losses.append(
            loss.item()
        )

        if batch_idx % 100 == 0:
            print(
                f"    batch {batch_idx:4d}/{len(loader)} "
                f"loss={loss.item():.4f}",
                flush=True,
            )

    return float(
        np.mean(
            losses
        )
    )


def train_cable_save_two_checkpoints(
    model,
    train_loader,
    val_loader,
    out_dir,
    run_metadata,
):
    """
    One 50-epoch CE training trajectory.

    Saves:
      1) best_val_dice.pt
      2) selected_dice_constrained_boundary.pt

    Test data are not touched here.
    """

    out_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    best_dice_path = (
        out_dir
        / "best_val_dice.pt"
    )

    boundary_path = (
        out_dir
        / "selected_dice_constrained_boundary.pt"
    )

    history_path = (
        out_dir
        / "history.csv"
    )

    optimizer = build_optimizer(
        model
    )

    scheduler = build_scheduler(
        optimizer
    )

    scaler = torch.cuda.amp.GradScaler(
        enabled=(
            cfg.use_amp
            and DEVICE.type == "cuda"
        )
    )

    history = []
    running_max_dice = -np.inf
    best_dice_epoch = -1

    # Candidate checkpoints remain local during the run.
    # Only the two final checkpoints are copied to Drive.
    temp_dir = Path(
        tempfile.mkdtemp(
            prefix="cable_boundary_candidates_"
        )
    )

    candidate_paths = {}

    start_time = time.time()

    try:
        for epoch in range(
            1,
            cfg.epochs + 1,
        ):
            train_loss = train_one_epoch(
                model,
                train_loader,
                optimizer,
                scaler,
            )

            val_summary, _ = evaluate_lesion_loader(
                model,
                val_loader,
            )

            row = {
                "epoch": int(
                    epoch
                ),
                "lr": float(
                    optimizer.param_groups[
                        0
                    ][
                        "lr"
                    ]
                ),
                "train_loss": float(
                    train_loss
                ),
                "val_loss": float(
                    val_summary[
                        "loss"
                    ]
                ),
                "val_dice": float(
                    val_summary[
                        "dice"
                    ]
                ),
                "val_iou": float(
                    val_summary[
                        "iou"
                    ]
                ),
                "val_hd95": float(
                    val_summary[
                        "hd95"
                    ]
                ),
                "val_boundary_iou": float(
                    val_summary[
                        "boundary_iou"
                    ]
                ),
            }

            history.append(
                row
            )

            # ------------------------------------------------
            # Ordinary best-Dice checkpoint — persisted to Drive
            # ------------------------------------------------
            if row[
                "val_dice"
            ] > running_max_dice:
                running_max_dice = row[
                    "val_dice"
                ]

                best_dice_epoch = int(
                    epoch
                )

                torch.save(
                    {
                        "model_state": model.state_dict(),
                        "epoch": int(
                            epoch
                        ),
                        "selection_policy": "max_validation_dice",
                        **run_metadata,
                    },
                    best_dice_path,
                )

            # ------------------------------------------------
            # Boundary-selection candidate checkpoint bookkeeping
            # ------------------------------------------------
            current_threshold = (
                running_max_dice
                - cfg.dice_tolerance
            )

            # Any checkpoint already below the current window can never
            # re-enter the final window because running_max_dice cannot decrease.
            stale_epochs = [
                ep
                for ep, meta
                in candidate_paths.items()
                if meta[
                    "val_dice"
                ]
                < current_threshold
            ]

            for ep in stale_epochs:
                p = candidate_paths[
                    ep
                ][
                    "path"
                ]

                if p.exists():
                    p.unlink()

                del candidate_paths[
                    ep
                ]

            if row[
                "val_dice"
            ] >= current_threshold:
                candidate_path = (
                    temp_dir
                    / f"epoch_{epoch:03d}.pt"
                )

                torch.save(
                    {
                        "model_state": model.state_dict(),
                        "epoch": int(
                            epoch
                        ),
                        "selection_policy": (
                            "dice_constrained_boundary_candidate"
                        ),
                        **run_metadata,
                    },
                    candidate_path,
                )

                candidate_paths[
                    int(
                        epoch
                    )
                ] = {
                    "path": candidate_path,
                    "val_dice": row[
                        "val_dice"
                    ],
                }

            # Persist the history after every epoch so the complete
            # validation trajectory is auditable even if Colab later fails.
            pd.DataFrame(
                history
            ).to_csv(
                history_path,
                index=False,
            )

            print(
                f"Epoch {epoch:02d}/{cfg.epochs} | "
                f"train={train_loss:.4f} | "
                f"Dice={row['val_dice']:.4f} | "
                f"IoU={row['val_iou']:.4f} | "
                f"HD95={row['val_hd95']:.2f} | "
                f"B-IoU={row['val_boundary_iou']:.4f}"
            )

            scheduler.step(
                row[
                    "val_loss"
                ]
            )

        history_df = pd.DataFrame(
            history
        )

        selection = select_boundary_checkpoint(
            history_df
        )

        boundary_epoch = int(
            selection[
                "selected_epoch"
            ]
        )

        assert best_dice_epoch == int(
            history_df.loc[
                history_df[
                    "val_dice"
                ].idxmax(),
                "epoch",
            ]
        )

        assert boundary_epoch in candidate_paths

        shutil.copy2(
            candidate_paths[
                boundary_epoch
            ][
                "path"
            ],
            boundary_path,
        )

        best_row = (
            history_df[
                history_df.epoch
                == best_dice_epoch
            ]
            .iloc[
                0
            ]
        )

        selection.update({
            "best_dice_epoch": int(
                best_dice_epoch
            ),
            "best_dice_val_dice": float(
                best_row.val_dice
            ),
            "best_dice_val_iou": float(
                best_row.val_iou
            ),
            "best_dice_val_hd95": float(
                best_row.val_hd95
            ),
            "best_dice_val_boundary_iou": float(
                best_row.val_boundary_iou
            ),
            "selected_differs_from_best_dice": bool(
                boundary_epoch
                != best_dice_epoch
            ),
            "train_seconds": float(
                time.time()
                - start_time
            ),
        })

        assert best_dice_path.exists()
        assert boundary_path.exists()

        return (
            history_df,
            selection,
            best_dice_path,
            boundary_path,
        )

    finally:
        shutil.rmtree(
            temp_dir,
            ignore_errors=True,
        )


In [44]:

# ============================================================
# 9. FRESH INFERENCE EFFICIENCY BENCHMARK
# ============================================================

@torch.inference_mode()
def benchmark_efficiency(
    model,
    warmup=30,
    runs=100,
    repeats=3,
):
    model.eval()

    row = {
        "latency_ms_median": np.nan,
        "latency_ms_q25": np.nan,
        "latency_ms_q75": np.nan,
        "peak_inference_memory_mb": np.nan,
    }

    if DEVICE.type != "cuda":
        return row

    x = torch.zeros(
        1,
        3,
        cfg.image_size,
        cfg.image_size,
        device=DEVICE,
    )

    for _ in range(
        warmup
    ):
        _ = model(
            x
        )

    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()

    times = []

    for _ in range(
        repeats
    ):
        for _ in range(
            runs
        ):
            start = torch.cuda.Event(
                enable_timing=True
            )

            end = torch.cuda.Event(
                enable_timing=True
            )

            start.record()
            _ = model(
                x
            )
            end.record()

            torch.cuda.synchronize()

            times.append(
                start.elapsed_time(
                    end
                )
            )

    row.update({
        "latency_ms_median": float(
            np.median(
                times
            )
        ),
        "latency_ms_q25": float(
            np.percentile(
                times,
                25,
            )
        ),
        "latency_ms_q75": float(
            np.percentile(
                times,
                75,
            )
        ),
        "peak_inference_memory_mb": float(
            torch.cuda.max_memory_allocated()
            / (1024 ** 2)
        ),
    })

    return row


efficiency_rows = []

for variant in ARCHITECTURE_VARIANTS:
    set_seed(
        12345
    )

    model = SwiftMSegFamily(
        variant=variant,
        num_classes=cfg.num_classes,
    ).to(
        DEVICE
    )

    efficiency_rows.append({
        "variant": variant,
        "params_M": count_trainable_params(
            model
        ) / 1e6,
        "aggregator_params_M": aggregator_params(
            model
        ) / 1e6,
        "aggregator_GMACs": complexity_lookup[
            variant
        ],
        **benchmark_efficiency(
            model
        ),
    })

    del model
    gc.collect()

    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()


efficiency_df = pd.DataFrame(
    efficiency_rows
)

display(
    efficiency_df
)

efficiency_df.to_csv(
    FINAL_ROOT
    / "EFFICIENCY_FINAL.csv",
    index=False,
)


Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

[transformers] SegformerModel LOAD REPORT from: nvidia/mit-b0
Key               | Status     |  | 
------------------+------------+--+-
classifier.weight | UNEXPECTED |  | 
classifier.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

[transformers] SegformerModel LOAD REPORT from: nvidia/mit-b0
Key               | Status     |  | 
------------------+------------+--+-
classifier.weight | UNEXPECTED |  | 
classifier.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

[transformers] SegformerModel LOAD REPORT from: nvidia/mit-b0
Key               | Status     |  | 
------------------+------------+--+-
classifier.weight | UNEXPECTED |  | 
classifier.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


,variant,params_M,aggregator_params_M,aggregator_GMACs,latency_ms_median,latency_ms_q25,latency_ms_q75,peak_inference_memory_mb
0,swift_lgl,4.485282,0.399360,1.312145,13.921840,12.009272,23.103505,47.674805
1,mhtm_hybrid_final,4.481858,0.395936,0.225690,13.944832,13.453712,15.849600,47.658691
2,cable_s1,4.487235,0.401313,0.231195,12.685904,12.395936,13.171128,47.681641


In [45]:

# ============================================================
# 10. AUTHORITATIVE BASELINE ARTIFACT AUDIT
# ============================================================

def bra_baseline_dir(
    fold,
    variant,
):
    return (
        OLD_BRA_AUDIT_ROOT
        / "official_cv"
        / f"fold_{fold}"
        / f"seed_{cfg.train_seed}"
        / variant
    )


def uclm_baseline_dir(
    fold,
    variant,
):
    return (
        OLD_UCLM_ROOT
        / "official_cv"
        / f"fold_{fold}"
        / f"seed_{cfg.train_seed}"
        / variant
    )


artifact_rows = []
missing_required = []


for fold in OUTER_FOLDS:
    for variant in BASELINE_VARIANTS:
        # BUS-BRA authoritative CLEAN audit.
        bra_dir = bra_baseline_dir(
            fold,
            variant,
        )

        bra_result = (
            bra_dir
            / "result_clean.json"
        )

        bra_per_image = (
            bra_dir
            / "per_image_test_clean.csv"
        )

        for kind, path in [
            (
                "BUS-BRA result_clean",
                bra_result,
            ),
            (
                "BUS-BRA per_image_test_clean",
                bra_per_image,
            ),
        ]:
            artifact_rows.append({
                "dataset": "BUS-BRA",
                "fold": fold,
                "variant": variant,
                "artifact": kind,
                "exists": path.exists(),
                "path": str(
                    path
                ),
            })

            if not path.exists():
                missing_required.append(
                    str(
                        path
                    )
                )

        # BUS-UCLM completed supervised experiment.
        uclm_dir = uclm_baseline_dir(
            fold,
            variant,
        )

        uclm_result = (
            uclm_dir
            / "result.json"
        )

        uclm_lesion = (
            uclm_dir
            / "per_image_test_lesion.csv"
        )

        uclm_normal = (
            uclm_dir
            / "per_image_test_normal.csv"
        )

        for kind, path in [
            (
                "BUS-UCLM result",
                uclm_result,
            ),
            (
                "BUS-UCLM lesion OOF",
                uclm_lesion,
            ),
            (
                "BUS-UCLM normal OOF",
                uclm_normal,
            ),
        ]:
            artifact_rows.append({
                "dataset": "BUS-UCLM",
                "fold": fold,
                "variant": variant,
                "artifact": kind,
                "exists": path.exists(),
                "path": str(
                    path
                ),
            })

            if not path.exists():
                missing_required.append(
                    str(
                        path
                    )
                )


artifact_audit = pd.DataFrame(
    artifact_rows
)

display(
    artifact_audit.groupby(
        [
            "dataset",
            "artifact",
        ]
    )[
        "exists"
    ].agg(
        [
            "sum",
            "count",
        ]
    )
)


if missing_required:
    print(
        "\nMissing authoritative baseline artifacts:"
    )

    for p in missing_required:
        print(
            " -",
            p,
        )

    raise FileNotFoundError(
        "Authoritative Swift/MHTM baseline artifacts are incomplete. "
        "This lean notebook intentionally will NOT retrain them."
    )


print(
    "\nPASS: all authoritative Swift-LGL and MHTM result/per-image "
    "artifacts required for the paper are present."
)


# Exact saved BUS-UCLM outer assignment is also required.
OLD_UCLM_ASSIGNMENT = (
    OLD_UCLM_ROOT
    / "BUSUCLM_OUTER_FOLD_ASSIGNMENT.csv"
)

assert OLD_UCLM_ASSIGNMENT.exists(), (
    "Missing exact previous BUS-UCLM outer-fold assignment:\n"
    f"{OLD_UCLM_ASSIGNMENT}"
)

print(
    "PASS: exact previous BUS-UCLM outer-fold assignment is available."
)


sum  count
dataset  artifact                                
BUS-BRA  BUS-BRA per_image_test_clean   10     10
         BUS-BRA result_clean           10     10
BUS-UCLM BUS-UCLM lesion OOF            10     10
         BUS-UCLM normal OOF            10     10
         BUS-UCLM result                10     10


PASS: all authoritative Swift-LGL and MHTM result/per-image artifacts required for the paper are present.
PASS: exact previous BUS-UCLM outer-fold assignment is available.


# Part A — BUS-BRA: five new CABLE trainings

In [46]:

# ============================================================
# 11. BUS-BRA DATA
# ============================================================

BUSBRA_ZIP = (
    DRIVE_ROOT
    / "BUSBRA.zip"
)

BUSBRA_LOCAL = (
    LOCAL_ROOT
    / "BUSBRA_DATA"
)

BUSBRA_URL = (
    "https://zenodo.org/records/8231412/"
    "files/BUSBRA.zip?download=1"
)


def download_busbra(
    url,
    dest,
):
    import requests
    from tqdm.auto import tqdm

    if (
        dest.exists()
        and dest.stat().st_size
        > 10_000_000
    ):
        print(
            "Using cached BUS-BRA ZIP:",
            dest,
        )
        return

    print(
        "Downloading BUS-BRA..."
    )

    with requests.get(
        url,
        stream=True,
        timeout=90,
    ) as r:
        r.raise_for_status()

        total = int(
            r.headers.get(
                "content-length",
                0,
            )
        )

        with open(
            dest,
            "wb",
        ) as f, tqdm(
            total=total,
            unit="B",
            unit_scale=True,
            desc=dest.name,
        ) as pbar:
            for chunk in r.iter_content(
                chunk_size=1024 * 1024
            ):
                if chunk:
                    f.write(
                        chunk
                    )

                    pbar.update(
                        len(
                            chunk
                        )
                    )


if RUN_BUSBRA:
    try:
        download_busbra(
            BUSBRA_URL,
            BUSBRA_ZIP,
        )

    except Exception as e:
        raise RuntimeError(
            "Automatic BUS-BRA download failed. "
            "Place BUSBRA.zip at:\n"
            f"{BUSBRA_ZIP}\n"
            f"Original error: {e}"
        )


    if not BUSBRA_LOCAL.exists():
        BUSBRA_LOCAL.mkdir(
            parents=True,
            exist_ok=True,
        )

        with zipfile.ZipFile(
            BUSBRA_ZIP,
            "r",
        ) as zf:
            zf.extractall(
                BUSBRA_LOCAL
            )


EXTRACT_ROOT = BUSBRA_LOCAL


Using cached BUS-BRA ZIP: /content/drive/MyDrive/MHTM_MEDICAL/BUSBRA.zip


In [47]:

# ============================================================
# 12. BUS-BRA OFFICIAL DATA AUDIT
# ============================================================

def find_named(
    root: Path,
    names,
    want_dir=False,
):
    names = {
        x.lower()
        for x in names
    }

    hits = []

    for p in root.rglob("*"):
        if p.name.lower() in names:
            if want_dir and p.is_dir():
                hits.append(p)

            elif (
                (not want_dir)
                and p.is_file()
            ):
                hits.append(p)

    if not hits:
        raise FileNotFoundError(
            f"Could not find {names} under {root}"
        )

    return sorted(
        hits,
        key=lambda p: len(p.parts),
    )[0]


IMAGES_DIR = find_named(
    EXTRACT_ROOT,
    ["Images"],
    want_dir=True,
)

MASKS_DIR = find_named(
    EXTRACT_ROOT,
    ["Masks", "Mask"],
    want_dir=True,
)

FOLDS_CSV = find_named(
    EXTRACT_ROOT,
    ["5-fold-cv.csv"],
    want_dir=False,
)


meta_df = pd.read_csv(
    FOLDS_CSV
)

required = {
    "ID",
    "Pathology",
    "kFold",
}

missing = required - set(
    meta_df.columns
)

assert not missing, (
    f"Missing columns: {missing}"
)


meta_df["Case"] = (
    meta_df["ID"]
    .str.extract(
        r"^(bus_\d+)"
    )[0]
)

assert (
    meta_df["Case"]
    .notna()
    .all()
)


meta_df["kFold"] = pd.to_numeric(
    meta_df["kFold"],
    errors="raise",
).astype(int)


def image_path(sample_id):
    candidates = [
        IMAGES_DIR / f"{sample_id}.png",
        IMAGES_DIR / f"{sample_id}.jpg",
        IMAGES_DIR / f"{sample_id}.jpeg",
    ]

    for p in candidates:
        if p.exists():
            return p

    raise FileNotFoundError(
        sample_id
    )


def mask_path(sample_id):
    suffix = sample_id.replace(
        "bus_",
        "",
    )

    candidates = [
        MASKS_DIR / f"mask_{suffix}.png",
        MASKS_DIR / f"{sample_id}_mask.png",
    ]

    for p in candidates:
        if p.exists():
            return p

    raise FileNotFoundError(
        sample_id
    )


assert len(meta_df) == 1875
assert meta_df["Case"].nunique() == 1064

assert sorted(
    meta_df["kFold"]
    .unique()
    .tolist()
) == [1, 2, 3, 4, 5]


# Verify every image-mask pair before training.
for sid in meta_df["ID"]:
    _ = image_path(sid)
    _ = mask_path(sid)


print("Images   :", len(meta_df))
print(
    "Patients :",
    meta_df["Case"].nunique(),
)
print("Images dir:", IMAGES_DIR)
print("Masks dir :", MASKS_DIR)

display(
    meta_df.head()
)


Images   : 1875
Patients : 1064
Images dir: /content/MHTM_MEDICAL/BUSBRA_DATA/BUSBRA/Images
Masks dir : /content/MHTM_MEDICAL/BUSBRA_DATA/BUSBRA/Masks


,ID,Pathology,kFold,valid_1,valid_2,valid_3,valid_4,valid_5,Case
0,bus_0001-l,malignant,3,1.0,1.0,NaN,1.0,1.0,bus_0001
1,bus_0001-r,malignant,3,1.0,1.0,NaN,1.0,1.0,bus_0001
2,bus_0002-l,benign,5,1.0,1.0,1.0,1.0,NaN,bus_0002
3,bus_0002-r,benign,5,1.0,1.0,0.0,1.0,NaN,bus_0002
4,bus_0003-l,malignant,4,0.0,1.0,0.0,NaN,1.0,bus_0003


In [48]:

# ============================================================
# 13. BUS-BRA OFFICIAL-FOLD SPLIT BUILDER
# ============================================================

# First verify that the supplied official folds themselves are patient-disjoint.
case_fold_count = (
    meta_df
    .groupby("Case")["kFold"]
    .nunique()
)

assert (
    case_fold_count.max() == 1
), (
    "Official BUS-BRA kFold assignment is not "
    "patient-disjoint under the parsed patient IDs."
)

print(
    "PASS: official BUS-BRA outer folds are patient-disjoint."
)


def make_official_cv_split(
    full_df,
    outer_fold,
    inner_seed,
):
    outer_fold = int(
        outer_fold
    )

    test_df = (
        full_df[
            full_df["kFold"]
            == outer_fold
        ]
        .copy()
        .reset_index(drop=True)
    )

    pool_df = (
        full_df[
            full_df["kFold"]
            != outer_fold
        ]
        .copy()
        .reset_index(drop=True)
    )

    inner = StratifiedGroupKFold(
        n_splits=5,
        shuffle=True,
        random_state=(
            inner_seed
            + outer_fold
        ),
    )

    train_idx, val_idx = next(
        inner.split(
            X=pool_df,
            y=pool_df["Pathology"],
            groups=pool_df["Case"],
        )
    )

    train_df = (
        pool_df
        .iloc[train_idx]
        .copy()
        .reset_index(drop=True)
    )

    val_df = (
        pool_df
        .iloc[val_idx]
        .copy()
        .reset_index(drop=True)
    )

    tr = set(train_df.Case)
    va = set(val_df.Case)
    te = set(test_df.Case)

    assert tr.isdisjoint(va)
    assert tr.isdisjoint(te)
    assert va.isdisjoint(te)

    assert (
        len(train_df)
        + len(val_df)
        + len(test_df)
        == len(full_df)
    )

    return (
        train_df,
        val_df,
        test_df,
    )


print("\nOfficial outer folds:")

display(
    meta_df
    .groupby("kFold")
    .agg(
        images=("ID", "size"),
        patients=("Case", "nunique"),
        malignant=(
            "Pathology",
            lambda s: (
                s.astype(str)
                .str.lower()
                == "malignant"
            ).sum(),
        ),
        benign=(
            "Pathology",
            lambda s: (
                s.astype(str)
                .str.lower()
                == "benign"
            ).sum(),
        ),
    )
)


print("\nFull strict split audit:")

split_audit_rows = []

for fold in OUTER_FOLDS:
    tr, va, te = make_official_cv_split(
        meta_df,
        fold,
        cfg.busbra_inner_val_seed,
    )

    row = {
        "fold": fold,
        "train_images": len(tr),
        "val_images": len(va),
        "test_images": len(te),
        "train_patients": tr.Case.nunique(),
        "val_patients": va.Case.nunique(),
        "test_patients": te.Case.nunique(),
    }

    split_audit_rows.append(
        row
    )

    print(
        f"Fold {fold}: "
        f"train={len(tr)} ({tr.Case.nunique()} patients) | "
        f"val={len(va)} ({va.Case.nunique()} patients) | "
        f"test={len(te)} ({te.Case.nunique()} patients)"
    )


split_audit_df = pd.DataFrame(
    split_audit_rows
)

display(
    split_audit_df
)


PASS: official BUS-BRA outer folds are patient-disjoint.

Official outer folds:


,images,patients,malignant,benign
kFold,,,,
1,376,212,122,254
2,385,213,125,260
3,366,213,120,246
4,365,214,119,246
5,383,212,121,262



Full strict split audit:
Fold 1: train=1200 (682 patients) | val=299 (170 patients) | test=376 (212 patients)
Fold 2: train=1196 (681 patients) | val=294 (170 patients) | test=385 (213 patients)
Fold 3: train=1206 (681 patients) | val=303 (170 patients) | test=366 (213 patients)
Fold 4: train=1210 (680 patients) | val=300 (170 patients) | test=365 (214 patients)
Fold 5: train=1194 (681 patients) | val=298 (171 patients) | test=383 (212 patients)


,fold,train_images,val_images,test_images,train_patients,val_patients,test_patients
0,1,1200,299,376,682,170,212
1,2,1196,294,385,681,170,213
2,3,1206,303,366,681,170,213
3,4,1210,300,365,680,170,214
4,5,1194,298,383,681,171,212


In [49]:

# ============================================================
# 14. BUS-BRA DATASET + TRANSFORMS
# ============================================================

def get_transforms(train=False):
    ops = [
        A.Resize(
            cfg.image_size,
            cfg.image_size,
        ),
    ]

    if train:
        ops += [
            A.HorizontalFlip(
                p=0.5
            ),
            A.RandomRotate90(
                p=0.5
            ),
        ]

    ops += [
        A.Normalize(
            mean=(
                0.485,
                0.456,
                0.406,
            ),
            std=(
                0.229,
                0.224,
                0.225,
            ),
        ),
        ToTensorV2(),
    ]

    return A.Compose(
        ops
    )


TRAIN_TRANSFORM = get_transforms(
    train=True
)

EVAL_TRANSFORM = get_transforms(
    train=False
)


class BUSBRADataset(Dataset):
    def __init__(
        self,
        df,
        train=False,
    ):
        self.df = (
            df
            .reset_index(drop=True)
        )

        self.transform = (
            TRAIN_TRANSFORM
            if train
            else EVAL_TRANSFORM
        )

    def __len__(self):
        return len(
            self.df
        )

    def __getitem__(self, idx):
        row = self.df.iloc[
            idx
        ]

        image = cv2.imread(
            str(image_path(row.ID)),
            cv2.IMREAD_COLOR,
        )

        if image is None:
            raise RuntimeError(
                f"Could not read image: {row.ID}"
            )

        image = cv2.cvtColor(
            image,
            cv2.COLOR_BGR2RGB,
        )

        mask = cv2.imread(
            str(mask_path(row.ID)),
            cv2.IMREAD_GRAYSCALE,
        )

        if mask is None:
            raise RuntimeError(
                f"Could not read mask: {row.ID}"
            )

        # Encoding adapter only.
        mask = (
            mask > 0
        ).astype(
            np.uint8
        )

        aug = self.transform(
            image=image,
            mask=mask,
        )

        image = (
            aug["image"]
            .float()
        )

        mask = (
            aug["mask"]
            .long()
        )

        meta = {
            "id": str(row.ID),
            "patient": str(row.Case),
            "pathology": str(row.Pathology).lower(),
        }

        return (
            image,
            mask,
            meta,
        )


def make_loaders(
    train_df,
    val_df,
    test_df,
    run_seed,
):
    train_ds = BUSBRADataset(
        train_df,
        train=True,
    )

    val_ds = BUSBRADataset(
        val_df,
        train=False,
    )

    test_ds = BUSBRADataset(
        test_df,
        train=False,
    )

    g = torch.Generator()
    g.manual_seed(
        run_seed
    )

    train_loader = DataLoader(
        train_ds,
        batch_size=cfg.batch_size,
        shuffle=True,
        num_workers=cfg.busbra_num_workers,
        pin_memory=(
            DEVICE.type == "cuda"
        ),
        drop_last=True,
        generator=g,
        worker_init_fn=seed_worker,
    )

    eval_kwargs = dict(
        batch_size=cfg.eval_batch_size,
        shuffle=False,
        num_workers=cfg.busbra_num_workers,
        pin_memory=(
            DEVICE.type == "cuda"
        ),
        drop_last=False,
        worker_init_fn=seed_worker,
    )

    val_loader = DataLoader(
        val_ds,
        **eval_kwargs,
    )

    test_loader = DataLoader(
        test_ds,
        **eval_kwargs,
    )

    return (
        train_loader,
        val_loader,
        test_loader,
    )


In [16]:

# ============================================================
# 14B. BUS-BRA PRE-FLIGHT INTERFACE CHECK
# ============================================================

_pre_tr, _pre_va, _pre_te = make_official_cv_split(
    meta_df,
    OUTER_FOLDS[0],
    cfg.busbra_inner_val_seed,
)

(
    _pre_train_loader,
    _pre_val_loader,
    _pre_test_loader,
) = make_loaders(
    _pre_tr,
    _pre_va,
    _pre_te,
    cfg.train_seed,
)

_pre_images, _pre_masks, _pre_meta = next(
    iter(
        _pre_val_loader
    )
)

assert _pre_images.ndim == 4
assert _pre_images.shape[1] == 3
assert _pre_masks.ndim == 3

assert set(
    _pre_meta.keys()
) == {
    "id",
    "patient",
    "pathology",
}, _pre_meta.keys()

assert len(
    _pre_meta["id"]
) == _pre_images.shape[0]

assert len(
    _pre_meta["patient"]
) == _pre_images.shape[0]

print(
    "PASS: BUS-BRA loader/evaluator interface is consistent."
)

print(
    "Example metadata:",
    {
        key: _pre_meta[key][0]
        for key in _pre_meta
    },
)

del (
    _pre_train_loader,
    _pre_val_loader,
    _pre_test_loader,
    _pre_images,
    _pre_masks,
    _pre_meta,
)


PASS: BUS-BRA loader/evaluator interface is consistent.
Example metadata: {'id': 'bus_0002-l', 'patient': 'bus_0002', 'pathology': 'benign'}


In [51]:

# ============================================================
# 15. BUS-BRA CABLE-ONLY FINAL RUN
# ============================================================

def busbra_cable_run_dir(
    fold,
):
    return (
        BUSBRA_FINAL_ROOT
        / f"fold_{fold}"
        / f"seed_{cfg.train_seed}"
        / CABLE_VARIANT
    )


def load_model_checkpoint(
    model,
    checkpoint_path,
):
    try:
        payload = torch.load(
            checkpoint_path,
            map_location=DEVICE,
            weights_only=False,
        )

    except TypeError:
        payload = torch.load(
            checkpoint_path,
            map_location=DEVICE,
        )

    incompatible = model.load_state_dict(
        payload[
            "model_state"
        ],
        strict=True,
    )

    assert len(
        incompatible.missing_keys
    ) == 0

    assert len(
        incompatible.unexpected_keys
    ) == 0

    return payload


def run_busbra_cable_fold(
    fold,
):
    out_dir = busbra_cable_run_dir(
        fold
    )

    out_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    result_path = (
        out_dir
        / "result_two_operating_points.json"
    )

    history_path = (
        out_dir
        / "history.csv"
    )

    selection_path = (
        out_dir
        / "selection.json"
    )

    best_dice_ckpt = (
        out_dir
        / "best_val_dice.pt"
    )

    boundary_ckpt = (
        out_dir
        / "selected_dice_constrained_boundary.pt"
    )

    dice_detail_path = (
        out_dir
        / "per_image_test_dice.csv"
    )

    boundary_detail_path = (
        out_dir
        / "per_image_test_boundary.csv"
    )

    dice_mask_path = (
        out_dir
        / "pred_masks_test_dice_256.npz"
    )

    boundary_mask_path = (
        out_dir
        / "pred_masks_test_boundary_256.npz"
    )

    complete = all(
        p.exists()
        for p in [
            result_path,
            history_path,
            selection_path,
            best_dice_ckpt,
            boundary_ckpt,
            dice_detail_path,
            boundary_detail_path,
            dice_mask_path,
            boundary_mask_path,
        ]
    )

    if (
        complete
        and not FORCE_RERUN
    ):
        print(
            f"REUSE BUS-BRA CABLE | fold={fold}"
        )

        with open(
            result_path,
            "r",
        ) as f:
            return json.load(
                f
            )


    train_df, val_df, test_df = make_official_cv_split(
        meta_df,
        fold,
        cfg.busbra_inner_val_seed,
    )

    set_seed(
        cfg.train_seed
    )

    (
        train_loader,
        val_loader,
        test_loader,
    ) = make_loaders(
        train_df,
        val_df,
        test_df,
        cfg.train_seed,
    )

    model = SwiftMSegFamily(
        variant=CABLE_VARIANT,
        num_classes=cfg.num_classes,
    ).to(
        DEVICE
    )

    run_metadata = {
        "dataset": "BUS-BRA",
        "outer_fold": int(
            fold
        ),
        "variant": CABLE_VARIANT,
        "train_seed": int(
            cfg.train_seed
        ),
    }

    print(
        "\n"
        + "=" * 100
    )

    print(
        f"TRAIN CABLE ONLY | BUS-BRA | fold={fold}"
    )

    print(
        "=" * 100
    )

    (
        history_df,
        selection,
        best_dice_path,
        boundary_path,
    ) = train_cable_save_two_checkpoints(
        model,
        train_loader,
        val_loader,
        out_dir,
        run_metadata,
    )

    history_df.to_csv(
        history_path,
        index=False,
    )

    with open(
        selection_path,
        "w",
    ) as f:
        json.dump(
            selection,
            f,
            indent=2,
        )


    # --------------------------------------------------------
    # 1) FAIR MAIN-TABLE OPERATING POINT: maximum validation Dice
    # --------------------------------------------------------
    payload_dice = load_model_checkpoint(
        model,
        best_dice_path,
    )

    dice_summary, dice_details = evaluate_lesion_loader(
        model,
        test_loader,
        save_masks_path=dice_mask_path,
    )

    dice_details = dice_details.assign(
        dataset="BUS-BRA",
        fold=int(
            fold
        ),
        train_seed=int(
            cfg.train_seed
        ),
        variant=CABLE_VARIANT,
        operating_point="dice",
    )

    dice_details.to_csv(
        dice_detail_path,
        index=False,
    )


    # --------------------------------------------------------
    # 2) ULTRASOUND BOUNDARY-AWARE OPERATING POINT
    # --------------------------------------------------------
    payload_boundary = load_model_checkpoint(
        model,
        boundary_path,
    )

    boundary_summary, boundary_details = evaluate_lesion_loader(
        model,
        test_loader,
        save_masks_path=boundary_mask_path,
    )

    boundary_details = boundary_details.assign(
        dataset="BUS-BRA",
        fold=int(
            fold
        ),
        train_seed=int(
            cfg.train_seed
        ),
        variant=CABLE_VARIANT,
        operating_point="boundary",
    )

    boundary_details.to_csv(
        boundary_detail_path,
        index=False,
    )


    result = {
        "dataset": "BUS-BRA",
        "fold": int(
            fold
        ),
        "variant": CABLE_VARIANT,
        "train_seed": int(
            cfg.train_seed
        ),

        "n_train_images": int(
            len(
                train_df
            )
        ),
        "n_val_images": int(
            len(
                val_df
            )
        ),
        "n_test_images": int(
            len(
                test_df
            )
        ),

        **selection,

        "dice_checkpoint_epoch": int(
            payload_dice[
                "epoch"
            ]
        ),

        "boundary_checkpoint_epoch": int(
            payload_boundary[
                "epoch"
            ]
        ),

        "dice_test_dice": dice_summary[
            "dice"
        ],
        "dice_test_iou": dice_summary[
            "iou"
        ],
        "dice_test_hd95": dice_summary[
            "hd95"
        ],
        "dice_test_boundary_iou": dice_summary[
            "boundary_iou"
        ],

        "boundary_test_dice": boundary_summary[
            "dice"
        ],
        "boundary_test_iou": boundary_summary[
            "iou"
        ],
        "boundary_test_hd95": boundary_summary[
            "hd95"
        ],
        "boundary_test_boundary_iou": boundary_summary[
            "boundary_iou"
        ],

        "params_M": float(
            architecture_audit.loc[
                architecture_audit.variant
                == CABLE_VARIANT,
                "params_M",
            ].iloc[
                0
            ]
        ),

        "aggregator_GMACs": float(
            complexity_lookup[
                CABLE_VARIANT
            ]
        ),

        "context_scale_dice_checkpoint": float(
            model.aggregators[
                0
            ].context_scale.detach().cpu()
        ),

        "best_dice_checkpoint": str(
            best_dice_path
        ),

        "boundary_checkpoint": str(
            boundary_path
        ),
    }


    with open(
        result_path,
        "w",
    ) as f:
        json.dump(
            result,
            f,
            indent=2,
        )


    print(
        json.dumps(
            {
                "fold": fold,
                "Dice checkpoint epoch": result[
                    "dice_checkpoint_epoch"
                ],
                "Boundary checkpoint epoch": result[
                    "boundary_checkpoint_epoch"
                ],
                "CABLE-Dice test Dice": result[
                    "dice_test_dice"
                ],
                "CABLE-Dice test HD95": result[
                    "dice_test_hd95"
                ],
                "CABLE-Boundary test Dice": result[
                    "boundary_test_dice"
                ],
                "CABLE-Boundary test HD95": result[
                    "boundary_test_hd95"
                ],
            },
            indent=2,
        )
    )


    del (
        model,
        train_loader,
        val_loader,
        test_loader,
        payload_dice,
        payload_boundary,
    )

    gc.collect()

    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()


    return result


In [18]:

# ============================================================
# 16. RUN THE FIVE BUS-BRA CABLE FOLDS
# ============================================================

busbra_cable_results = []

if RUN_BUSBRA:
    for fold in OUTER_FOLDS:
        busbra_cable_results.append(
            run_busbra_cable_fold(
                fold
            )
        )

    busbra_cable_results_df = (
        pd.DataFrame(
            busbra_cable_results
        )
        .sort_values(
            "fold"
        )
        .reset_index(
            drop=True
        )
    )

    assert len(
        busbra_cable_results_df
    ) == 5

    busbra_cable_results_df.to_csv(
        BUSBRA_FINAL_ROOT
        / "CABLE_ALL_FOLD_RESULTS.csv",
        index=False,
    )

    display(
        busbra_cable_results_df[
            [
                "fold",
                "dice_checkpoint_epoch",
                "boundary_checkpoint_epoch",
                "selected_differs_from_best_dice",
                "dice_test_dice",
                "dice_test_hd95",
                "boundary_test_dice",
                "boundary_test_hd95",
            ]
        ]
    )


Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

[transformers] SegformerModel LOAD REPORT from: nvidia/mit-b0
Key               | Status     |  | 
------------------+------------+--+-
classifier.weight | UNEXPECTED |  | 
classifier.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



TRAIN CABLE ONLY | BUS-BRA | fold=1
    batch    0/600 loss=0.6981
    batch  100/600 loss=0.2226
    batch  200/600 loss=0.2680
    batch  300/600 loss=0.2906
    batch  400/600 loss=0.1800
    batch  500/600 loss=0.0926
Epoch 01/50 | train=0.2403 | Dice=0.0069 | IoU=0.0038 | HD95=309.71 | B-IoU=0.0003
    batch    0/600 loss=0.1743
    batch  100/600 loss=0.1507
    batch  200/600 loss=0.2342
    batch  300/600 loss=0.1413
    batch  400/600 loss=0.1416
    batch  500/600 loss=0.1285
Epoch 02/50 | train=0.1990 | Dice=0.2552 | IoU=0.1878 | HD95=118.25 | B-IoU=0.0500
    batch    0/600 loss=0.1532
    batch  100/600 loss=0.3042
    batch  200/600 loss=0.3022
    batch  300/600 loss=0.0859
    batch  400/600 loss=0.1154
    batch  500/600 loss=0.0499
Epoch 03/50 | train=0.1706 | Dice=0.6447 | IoU=0.5174 | HD95=41.07 | B-IoU=0.1391
    batch    0/600 loss=0.0924
    batch  100/600 loss=0.2118
    batch  200/600 loss=0.1238
    batch  300/600 loss=0.0622
    batch  400/600 loss=0.0597
  

Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

[transformers] SegformerModel LOAD REPORT from: nvidia/mit-b0
Key               | Status     |  | 
------------------+------------+--+-
classifier.weight | UNEXPECTED |  | 
classifier.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...


TRAIN CABLE ONLY | BUS-BRA | fold=4
    batch    0/605 loss=0.6949
    batch  100/605 loss=0.2238
    batch  200/605 loss=0.1584
    batch  300/605 loss=0.9008
    batch  400/605 loss=0.2810
    batch  500/605 loss=0.2990
    batch  600/605 loss=0.1409
Epoch 01/50 | train=0.2378 | Dice=0.3798 | IoU=0.2631 | HD95=65.09 | B-IoU=0.0471
    batch    0/605 loss=0.1492
    batch  100/605 loss=0.2117
    batch  200/605 loss=0.4923
    batch  300/605 loss=0.1047
    batch  400/605 loss=0.1661
    batch  500/605 loss=0.0375
    batch  600/605 loss=0.2558
Epoch 02/50 | train=0.1954 | Dice=0.4391 | IoU=0.3177 | HD95=51.41 | B-IoU=0.0584
    batch    0/605 loss=0.2293
    batch  100/605 loss=0.0787
    batch  200/605 loss=0.0976
    batch  300/605 loss=0.2412
    batch  400/605 loss=0.1039
    batch  500/605 loss=0.1526
    batch  600/605 loss=0.2161
Epoch 03/50 | train=0.1685 | Dice=0.6667 | IoU=0.5419 | HD95=47.73 | B-IoU=0.1489
    batch    0/605 loss=0.0813
    batch  100/605 loss=0.0715
    

Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

[transformers] SegformerModel LOAD REPORT from: nvidia/mit-b0
Key               | Status     |  | 
------------------+------------+--+-
classifier.weight | UNEXPECTED |  | 
classifier.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



TRAIN CABLE ONLY | BUS-BRA | fold=5
    batch    0/597 loss=0.6991
    batch  100/597 loss=0.3072
    batch  200/597 loss=0.1217
    batch  300/597 loss=0.3127
    batch  400/597 loss=0.1276
    batch  500/597 loss=0.1684
Epoch 01/50 | train=0.2374 | Dice=0.0083 | IoU=0.0045 | HD95=300.51 | B-IoU=0.0012
    batch    0/597 loss=0.4046
    batch  100/597 loss=0.3430
    batch  200/597 loss=0.2837
    batch  300/597 loss=0.0960
    batch  400/597 loss=0.1511
    batch  500/597 loss=0.1541
Epoch 02/50 | train=0.1927 | Dice=0.1433 | IoU=0.0960 | HD95=191.56 | B-IoU=0.0196
    batch    0/597 loss=0.2296
    batch  100/597 loss=0.1508
    batch  200/597 loss=0.1358
    batch  300/597 loss=0.1285
    batch  400/597 loss=0.1360
    batch  500/597 loss=0.1823
Epoch 03/50 | train=0.2125 | Dice=0.0047 | IoU=0.0026 | HD95=340.54 | B-IoU=0.0002
    batch    0/597 loss=0.3368
    batch  100/597 loss=0.1181
    batch  200/597 loss=0.1512
    batch  300/597 loss=0.0992
    batch  400/597 loss=0.1445
 

,fold,dice_checkpoint_epoch,boundary_checkpoint_epoch,selected_differs_from_best_dice,dice_test_dice,dice_test_hd95,boundary_test_dice,boundary_test_hd95
0,1,45,48,True,0.855043,17.635295,0.857781,16.108930
1,2,39,49,True,0.838175,20.051138,0.840221,20.301026
2,3,50,50,False,0.856073,17.445600,0.856073,17.445600
3,4,32,30,True,0.845700,20.106434,0.837718,18.354040
4,5,50,48,True,0.847469,19.520115,0.844668,19.228618


# Part B — BUS-UCLM: five new CABLE trainings

In [52]:

# ============================================================
# 17. BUS-UCLM DATA CACHE
# ============================================================

BUSUCLM_DRIVE = (
    DRIVE_ROOT
    / "BUS_UCLM"
)

BUSUCLM_ZIP = (
    BUSUCLM_DRIVE
    / "7fvgj4jsp7-1.zip"
)

BUSUCLM_LOCAL = (
    LOCAL_ROOT
    / "BUS_UCLM_DATA"
)

if RUN_BUSUCLM:
    assert BUSUCLM_ZIP.exists(), (
        "Place official BUS-UCLM ZIP at:\n"
        f"{BUSUCLM_ZIP}"
    )

    already_extracted = (
        BUSUCLM_LOCAL.exists()
        and any(
            BUSUCLM_LOCAL.rglob(
                "*.png"
            )
        )
    )

    if not already_extracted:
        BUSUCLM_LOCAL.mkdir(
            parents=True,
            exist_ok=True,
        )

        with zipfile.ZipFile(
            BUSUCLM_ZIP,
            "r",
        ) as zf:
            bad = zf.testzip()

            if bad is not None:
                raise RuntimeError(
                    f"ZIP integrity failure: {bad}"
                )

            zf.extractall(
                BUSUCLM_LOCAL
            )

    print(
        "BUS-UCLM available at:",
        BUSUCLM_LOCAL,
    )


UCLM_FINAL_ROOT = (
    FINAL_ROOT
    / "BUS_UCLM"
)

UCLM_FINAL_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


BUS-UCLM available at: /content/MHTM_MEDICAL/BUS_UCLM_DATA


In [53]:

# ============================================================
# 18. BUS-UCLM PUBLIC MANIFEST
# ============================================================

def candidate_named_dirs(
    root,
    target_name,
):
    return [
        p
        for p in root.rglob("*")
        if p.is_dir()
        and p.name.lower()
        == target_name.lower()
    ]


image_dirs = candidate_named_dirs(
    BUSUCLM_LOCAL,
    "images",
)

mask_dirs = candidate_named_dirs(
    BUSUCLM_LOCAL,
    "masks",
)

if (
    not image_dirs
    or not mask_dirs
):
    raise FileNotFoundError(
        "Could not find BUS-UCLM images/ and masks/ directories."
    )


best = None

for idir in image_dirs:
    image_files = {
        p.stem: p
        for p in idir.glob("*.png")
    }

    for mdir in mask_dirs:
        mask_files = {
            p.stem: p
            for p in mdir.glob("*.png")
        }

        common = (
            set(image_files)
            & set(mask_files)
        )

        score = len(common)

        if (
            best is None
            or score > best[0]
        ):
            best = (
                score,
                idir,
                mdir,
                image_files,
                mask_files,
            )


(
    score,
    IMAGES_DIR,
    MASKS_DIR,
    image_files,
    mask_files,
) = best

assert score == 683, (
    f"Expected 683 matched image/mask filenames, found {score}"
)


def inspect_rgb_mask(
    mask_path,
):
    bgr = cv2.imread(
        str(mask_path),
        cv2.IMREAD_COLOR,
    )

    if bgr is None:
        raise RuntimeError(
            f"Could not read mask: {mask_path}"
        )

    rgb = cv2.cvtColor(
        bgr,
        cv2.COLOR_BGR2RGB,
    )

    R = rgb[..., 0].astype(np.int16)
    G = rgb[..., 1].astype(np.int16)
    B = rgb[..., 2].astype(np.int16)

    lesion = (
        (R > 0)
        | (G > 0)
        | (B > 0)
    )

    if not lesion.any():
        pathology = "normal"

    else:
        red_score = int(
            np.logical_and(
                R > G,
                R > B,
            ).sum()
        )

        green_score = int(
            np.logical_and(
                G > R,
                G > B,
            ).sum()
        )

        if red_score > green_score:
            pathology = "malignant"

        elif green_score > red_score:
            pathology = "benign"

        else:
            raise RuntimeError(
                f"Cannot infer mask pathology: {mask_path}"
            )

    return (
        pathology,
        int(
            lesion.sum()
        ),
    )


manifest_rows = []

for stem in sorted(
    set(image_files)
    & set(mask_files)
):
    pathology, lesion_pixels = (
        inspect_rgb_mask(
            mask_files[stem]
        )
    )

    patient = (
        stem.split("_")[0]
    )

    manifest_rows.append({
        "id": stem,
        "patient": patient,
        "pathology": pathology,
        "image_path": str(
            image_files[stem]
        ),
        "mask_path": str(
            mask_files[stem]
        ),
        "lesion_pixels_original": lesion_pixels,
    })


manifest_public = pd.DataFrame(
    manifest_rows
)

assert len(manifest_public) == 683
assert manifest_public.patient.nunique() == 38

public_counts = (
    manifest_public.pathology
    .value_counts()
    .to_dict()
)

assert public_counts == {
    "normal": 419,
    "benign": 174,
    "malignant": 90,
}, public_counts

print(
    "Public BUS-UCLM manifest: PASS"
)

display(
    manifest_public.groupby(
        "pathology"
    ).agg(
        images=("id", "size"),
        patients=("patient", "nunique"),
    )
)


Public BUS-UCLM manifest: PASS


,images,patients
pathology,,
benign,174,24
malignant,90,17
normal,419,35


In [54]:

# ============================================================
# 19. BUS-UCLM GEOMETRY AUDIT
# ============================================================

shape_rows = []

for row in manifest_public.itertuples(
    index=False
):
    image = cv2.imread(
        row.image_path,
        cv2.IMREAD_COLOR,
    )

    mask = cv2.imread(
        row.mask_path,
        cv2.IMREAD_COLOR,
    )

    if (
        image is None
        or mask is None
    ):
        raise RuntimeError(
            f"Read failure: {row.id}"
        )

    ih, iw = image.shape[:2]
    mh, mw = mask.shape[:2]

    shape_rows.append({
        "id": row.id,
        "image_h": ih,
        "image_w": iw,
        "mask_h": mh,
        "mask_w": mw,
        "same_shape": (
            ih == mh
            and iw == mw
        ),
    })


shape_audit = pd.DataFrame(
    shape_rows
)

manifest_full = (
    manifest_public
    .merge(
        shape_audit,
        on="id",
        how="left",
        validate="one_to_one",
    )
)

excluded_geometry = (
    manifest_full[
        ~manifest_full.same_shape
    ]
    .copy()
    .reset_index(drop=True)
)

manifest = (
    manifest_full[
        manifest_full.same_shape
    ]
    .copy()
    .reset_index(drop=True)
)


assert len(excluded_geometry) == 13
assert len(manifest) == 670
assert manifest.patient.nunique() == 37

counts = (
    manifest.pathology
    .value_counts()
    .to_dict()
)

assert counts == {
    "normal": 410,
    "benign": 170,
    "malignant": 90,
}, counts


print(
    "Geometry-valid evaluation/training pool: PASS"
)

print(
    "Retained:",
    len(manifest),
    "images from",
    manifest.patient.nunique(),
    "patients",
)

print(
    "Excluded:",
    len(excluded_geometry),
    "incompatible public pairs",
)

display(
    manifest.groupby(
        "pathology"
    ).agg(
        images=("id", "size"),
        patients=("patient", "nunique"),
    )
)


manifest.to_csv(
    UCLM_FINAL_ROOT
    / "BUSUCLM_GEOMETRY_VALID_MANIFEST_670.csv",
    index=False,
)

excluded_geometry.to_csv(
    UCLM_FINAL_ROOT
    / "BUSUCLM_EXCLUDED_GEOMETRY_MISMATCHES.csv",
    index=False,
)


Geometry-valid evaluation/training pool: PASS
Retained: 670 images from 37 patients
Excluded: 13 incompatible public pairs


,images,patients
pathology,,
benign,170,23
malignant,90,17
normal,410,34


In [55]:

# ============================================================
# 20. BUS-UCLM — LOAD EXACT PREVIOUS OUTER-FOLD ASSIGNMENT
# ============================================================

saved_assignment = pd.read_csv(
    OLD_UCLM_ASSIGNMENT
)

manifest = manifest.copy()

manifest[
    "id"
] = manifest[
    "id"
].astype(
    str
)

saved_assignment[
    "id"
] = saved_assignment[
    "id"
].astype(
    str
)


required_saved_cols = {
    "id",
    "outer_fold",
}

assert required_saved_cols.issubset(
    saved_assignment.columns
)


manifest = (
    manifest.drop(
        columns=[
            "outer_fold",
        ],
        errors="ignore",
    )
    .merge(
        saved_assignment[
            [
                "id",
                "outer_fold",
            ]
        ],
        on="id",
        how="left",
        validate="one_to_one",
    )
)


assert len(
    manifest
) == 670

assert manifest[
    "outer_fold"
].notna().all()

manifest[
    "outer_fold"
] = manifest[
    "outer_fold"
].astype(
    int
)

assert manifest[
    "outer_fold"
].between(
    1,
    5,
).all()


patient_fold_count = (
    manifest.groupby(
        "patient"
    )[
        "outer_fold"
    ]
    .nunique()
)

assert patient_fold_count.max() == 1


outer_fold_audit = (
    manifest.groupby(
        "outer_fold"
    )
    .agg(
        images=("id", "size"),
        patients=("patient", "nunique"),
        benign=(
            "pathology",
            lambda s: (
                s == "benign"
            ).sum(),
        ),
        malignant=(
            "pathology",
            lambda s: (
                s == "malignant"
            ).sum(),
        ),
        normal=(
            "pathology",
            lambda s: (
                s == "normal"
            ).sum(),
        ),
    )
)


display(
    outer_fold_audit
)

print(
    "PASS: exact saved BUS-UCLM outer folds from the "
    "authoritative baseline experiment were reused."
)


manifest.to_csv(
    UCLM_FINAL_ROOT
    / "BUSUCLM_OUTER_FOLD_ASSIGNMENT_REUSED.csv",
    index=False,
)


,images,patients,benign,malignant,normal
outer_fold,,,,,
1,123,8,37,13,73
2,165,7,47,25,93
3,151,8,50,19,82
4,110,7,29,6,75
5,121,7,7,27,87


PASS: exact saved BUS-UCLM outer folds from the authoritative baseline experiment were reused.


In [56]:

# ============================================================
# 21. BUS-UCLM INNER VALIDATION SPLIT
# ============================================================

def make_split(
    outer_fold,
):
    outer_fold = int(
        outer_fold
    )

    test_all = (
        manifest[
            manifest.outer_fold
            == outer_fold
        ]
        .copy()
        .reset_index(drop=True)
    )

    dev_all = (
        manifest[
            manifest.outer_fold
            != outer_fold
        ]
        .copy()
        .reset_index(drop=True)
    )


    # Primary training task is lesion segmentation.
    dev_lesion = (
        dev_all[
            dev_all.pathology
            .isin(
                [
                    "benign",
                    "malignant",
                ]
            )
        ]
        .copy()
        .reset_index(drop=True)
    )


    inner = StratifiedGroupKFold(
        n_splits=5,
        shuffle=True,
        random_state=(
            cfg.uclm_inner_val_seed
            + outer_fold
        ),
    )


    # Choose a deterministic candidate split using labels/counts only.
    # This is not model-performance selection.
    candidates = []

    overall_prop = (
        dev_lesion.pathology
        .value_counts(
            normalize=True
        )
    )

    for split_idx, (
        train_idx,
        val_idx,
    ) in enumerate(
        inner.split(
            X=dev_lesion,
            y=dev_lesion.pathology,
            groups=dev_lesion.patient,
        )
    ):
        tr = (
            dev_lesion.iloc[
                train_idx
            ]
            .copy()
        )

        va = (
            dev_lesion.iloc[
                val_idx
            ]
            .copy()
        )

        # Require both lesion pathologies in train and validation.
        if (
            tr.pathology.nunique()
            < 2
            or va.pathology.nunique()
            < 2
        ):
            continue

        val_prop = (
            va.pathology
            .value_counts(
                normalize=True
            )
        )

        balance_penalty = sum(
            abs(
                float(
                    val_prop.get(
                        cls,
                        0.0,
                    )
                )
                - float(
                    overall_prop.get(
                        cls,
                        0.0,
                    )
                )
            )
            for cls in [
                "benign",
                "malignant",
            ]
        )

        size_penalty = abs(
            len(va)
            / len(dev_lesion)
            - 0.20
        )

        candidates.append(
            (
                balance_penalty
                + size_penalty,
                split_idx,
                train_idx,
                val_idx,
            )
        )


    if not candidates:
        raise RuntimeError(
            f"No valid inner split for outer fold {outer_fold}"
        )


    candidates.sort(
        key=lambda x: (
            x[0],
            x[1],
        )
    )

    _, selected_idx, train_idx, val_idx = (
        candidates[0]
    )


    train_df = (
        dev_lesion.iloc[
            train_idx
        ]
        .copy()
        .reset_index(drop=True)
    )

    val_df = (
        dev_lesion.iloc[
            val_idx
        ]
        .copy()
        .reset_index(drop=True)
    )

    test_lesion = (
        test_all[
            test_all.pathology
            .isin(
                [
                    "benign",
                    "malignant",
                ]
            )
        ]
        .copy()
        .reset_index(drop=True)
    )

    test_normal = (
        test_all[
            test_all.pathology
            == "normal"
        ]
        .copy()
        .reset_index(drop=True)
    )


    tr_pat = set(
        train_df.patient
    )
    va_pat = set(
        val_df.patient
    )
    te_pat = set(
        test_all.patient
    )

    assert tr_pat.isdisjoint(
        va_pat
    )

    assert tr_pat.isdisjoint(
        te_pat
    )

    assert va_pat.isdisjoint(
        te_pat
    )

    assert len(
        test_lesion
    ) > 0

    assert len(
        test_normal
    ) > 0


    return {
        "train": train_df,
        "val": val_df,
        "test_lesion": test_lesion,
        "test_normal": test_normal,
        "test_all": test_all,
        "inner_split_index": int(
            selected_idx
        ),
    }


split_rows = []

for fold in OUTER_FOLDS:
    s = make_split(
        fold
    )

    split_rows.append({
        "fold": fold,
        "inner_split_index": s[
            "inner_split_index"
        ],

        "train_lesion_images": len(
            s["train"]
        ),
        "train_patients": s[
            "train"
        ].patient.nunique(),

        "val_lesion_images": len(
            s["val"]
        ),
        "val_patients": s[
            "val"
        ].patient.nunique(),

        "test_lesion_images": len(
            s["test_lesion"]
        ),
        "test_normal_images": len(
            s["test_normal"]
        ),
        "test_patients_all": s[
            "test_all"
        ].patient.nunique(),

        "train_malignant": int(
            (
                s["train"].pathology
                == "malignant"
            ).sum()
        ),
        "val_malignant": int(
            (
                s["val"].pathology
                == "malignant"
            ).sum()
        ),
        "test_malignant": int(
            (
                s["test_lesion"].pathology
                == "malignant"
            ).sum()
        ),
    })


split_audit = pd.DataFrame(
    split_rows
)

display(
    split_audit
)

split_audit.to_csv(
    UCLM_FINAL_ROOT
    / "BUSUCLM_SUPERVISED_SPLIT_AUDIT.csv",
    index=False,
)


,fold,inner_split_index,train_lesion_images,train_patients,val_lesion_images,val_patients,test_lesion_images,test_normal_images,test_patients_all,train_malignant,val_malignant,test_malignant
0,1,1,173,22,37,5,50,73,8,63,14,13
1,2,1,150,21,38,7,72,93,7,53,12,25
2,3,3,136,21,55,6,69,82,8,53,18,19
3,4,3,159,24,66,6,35,75,7,56,28,6
4,5,1,192,23,34,5,34,87,7,55,8,27


In [57]:

# ============================================================
# 22. BUS-UCLM DATASET + LOADERS
# ============================================================

UCLM_TRAIN_TRANSFORM = A.Compose([
    A.Resize(
        cfg.image_size,
        cfg.image_size,
    ),
    A.HorizontalFlip(
        p=0.5
    ),
    A.RandomRotate90(
        p=0.5
    ),
    A.Normalize(
        mean=(
            0.485,
            0.456,
            0.406,
        ),
        std=(
            0.229,
            0.224,
            0.225,
        ),
    ),
    ToTensorV2(),
])


UCLM_EVAL_TRANSFORM = A.Compose([
    A.Resize(
        cfg.image_size,
        cfg.image_size,
    ),
    A.Normalize(
        mean=(
            0.485,
            0.456,
            0.406,
        ),
        std=(
            0.229,
            0.224,
            0.225,
        ),
    ),
    ToTensorV2(),
])


class BUSUCLMDataset(Dataset):
    def __init__(
        self,
        df,
        train=False,
    ):
        self.df = (
            df
            .reset_index(
                drop=True
            )
        )

        self.transform = (
            UCLM_TRAIN_TRANSFORM
            if train
            else UCLM_EVAL_TRANSFORM
        )

    def __len__(
        self
    ):
        return len(
            self.df
        )

    def __getitem__(
        self,
        idx,
    ):
        row = self.df.iloc[
            idx
        ]

        image = cv2.imread(
            row.image_path,
            cv2.IMREAD_COLOR,
        )

        mask_rgb = cv2.imread(
            row.mask_path,
            cv2.IMREAD_COLOR,
        )

        if (
            image is None
            or mask_rgb is None
        ):
            raise RuntimeError(
                f"Read failure: {row.id}"
            )

        image = cv2.cvtColor(
            image,
            cv2.COLOR_BGR2RGB,
        )

        mask_rgb = cv2.cvtColor(
            mask_rgb,
            cv2.COLOR_BGR2RGB,
        )

        assert (
            image.shape[:2]
            == mask_rgb.shape[:2]
        )

        mask = (
            np.any(
                mask_rgb > 0,
                axis=-1,
            )
        ).astype(
            np.uint8
        )

        aug = self.transform(
            image=image,
            mask=mask,
        )

        return (
            aug["image"].float(),
            aug["mask"].long(),
            {
                "id": str(
                    row.id
                ),
                "patient": str(
                    row.patient
                ),
                "pathology": str(
                    row.pathology
                ),
            },
        )


def make_uclm_loaders(
    split,
    run_seed,
):
    train_ds = BUSUCLMDataset(
        split[
            "train"
        ],
        train=True,
    )

    val_ds = BUSUCLMDataset(
        split[
            "val"
        ],
        train=False,
    )

    test_lesion_ds = BUSUCLMDataset(
        split[
            "test_lesion"
        ],
        train=False,
    )

    test_normal_ds = BUSUCLMDataset(
        split[
            "test_normal"
        ],
        train=False,
    )

    g = torch.Generator()
    g.manual_seed(
        run_seed
    )

    train_loader = DataLoader(
        train_ds,
        batch_size=cfg.batch_size,
        shuffle=True,
        num_workers=cfg.uclm_num_workers,
        pin_memory=(
            DEVICE.type == "cuda"
        ),
        drop_last=True,
        generator=g,
        worker_init_fn=seed_worker,
    )

    eval_kwargs = dict(
        batch_size=cfg.eval_batch_size,
        shuffle=False,
        num_workers=cfg.uclm_num_workers,
        pin_memory=(
            DEVICE.type == "cuda"
        ),
        drop_last=False,
        worker_init_fn=seed_worker,
    )

    return (
        train_loader,
        DataLoader(
            val_ds,
            **eval_kwargs,
        ),
        DataLoader(
            test_lesion_ds,
            **eval_kwargs,
        ),
        DataLoader(
            test_normal_ds,
            **eval_kwargs,
        ),
    )


In [58]:

# ============================================================
# 22B. BUS-UCLM PRE-FLIGHT INTERFACE CHECK
# ============================================================

_pre_split = make_split(
    OUTER_FOLDS[0]
)

(
    _pre_train_loader,
    _pre_val_loader,
    _pre_test_lesion_loader,
    _pre_test_normal_loader,
) = make_uclm_loaders(
    _pre_split,
    cfg.train_seed,
)

_pre_images, _pre_masks, _pre_meta = next(
    iter(
        _pre_val_loader
    )
)

assert _pre_images.ndim == 4
assert _pre_images.shape[1] == 3
assert _pre_masks.ndim == 3

assert set(
    _pre_meta.keys()
) == {
    "id",
    "patient",
    "pathology",
}, _pre_meta.keys()

print(
    "PASS: BUS-UCLM loader/evaluator interface is consistent."
)

print(
    "Example metadata:",
    {
        key: _pre_meta[key][0]
        for key in _pre_meta
    },
)

del (
    _pre_train_loader,
    _pre_val_loader,
    _pre_test_lesion_loader,
    _pre_test_normal_loader,
    _pre_images,
    _pre_masks,
    _pre_meta,
    _pre_split,
)


PASS: BUS-UCLM loader/evaluator interface is consistent.
Example metadata: {'id': 'CHSP_006', 'patient': 'CHSP', 'pathology': 'malignant'}


In [59]:

# ============================================================
# 23. BUS-UCLM CABLE-ONLY FINAL RUN
# ============================================================

def uclm_cable_run_dir(
    fold,
):
    return (
        UCLM_FINAL_ROOT
        / f"fold_{fold}"
        / f"seed_{cfg.train_seed}"
        / CABLE_VARIANT
    )


def run_uclm_cable_fold(
    fold,
):
    out_dir = uclm_cable_run_dir(
        fold
    )

    out_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    result_path = (
        out_dir
        / "result_two_operating_points.json"
    )

    history_path = (
        out_dir
        / "history.csv"
    )

    selection_path = (
        out_dir
        / "selection.json"
    )

    best_dice_ckpt = (
        out_dir
        / "best_val_dice.pt"
    )

    boundary_ckpt = (
        out_dir
        / "selected_dice_constrained_boundary.pt"
    )

    dice_lesion_path = (
        out_dir
        / "per_image_test_lesion_dice.csv"
    )

    boundary_lesion_path = (
        out_dir
        / "per_image_test_lesion_boundary.csv"
    )

    dice_normal_path = (
        out_dir
        / "per_image_test_normal_dice.csv"
    )

    boundary_normal_path = (
        out_dir
        / "per_image_test_normal_boundary.csv"
    )

    dice_lesion_mask = (
        out_dir
        / "pred_masks_test_lesion_dice_256.npz"
    )

    boundary_lesion_mask = (
        out_dir
        / "pred_masks_test_lesion_boundary_256.npz"
    )

    dice_normal_mask = (
        out_dir
        / "pred_masks_test_normal_dice_256.npz"
    )

    boundary_normal_mask = (
        out_dir
        / "pred_masks_test_normal_boundary_256.npz"
    )

    complete = all(
        p.exists()
        for p in [
            result_path,
            history_path,
            selection_path,
            best_dice_ckpt,
            boundary_ckpt,
            dice_lesion_path,
            boundary_lesion_path,
            dice_normal_path,
            boundary_normal_path,
            dice_lesion_mask,
            boundary_lesion_mask,
            dice_normal_mask,
            boundary_normal_mask,
        ]
    )

    if (
        complete
        and not FORCE_RERUN
    ):
        print(
            f"REUSE BUS-UCLM CABLE | fold={fold}"
        )

        with open(
            result_path,
            "r",
        ) as f:
            return json.load(
                f
            )


    split = make_split(
        fold
    )

    set_seed(
        cfg.train_seed
    )

    (
        train_loader,
        val_loader,
        test_lesion_loader,
        test_normal_loader,
    ) = make_uclm_loaders(
        split,
        cfg.train_seed,
    )

    model = SwiftMSegFamily(
        variant=CABLE_VARIANT,
        num_classes=cfg.num_classes,
    ).to(
        DEVICE
    )

    run_metadata = {
        "dataset": "BUS-UCLM",
        "outer_fold": int(
            fold
        ),
        "variant": CABLE_VARIANT,
        "train_seed": int(
            cfg.train_seed
        ),
    }

    print(
        "\n"
        + "=" * 100
    )

    print(
        f"TRAIN CABLE ONLY | BUS-UCLM | fold={fold}"
    )

    print(
        "=" * 100
    )

    (
        history_df,
        selection,
        best_dice_path,
        boundary_path,
    ) = train_cable_save_two_checkpoints(
        model,
        train_loader,
        val_loader,
        out_dir,
        run_metadata,
    )

    history_df.to_csv(
        history_path,
        index=False,
    )

    with open(
        selection_path,
        "w",
    ) as f:
        json.dump(
            selection,
            f,
            indent=2,
        )


    # --------------------------------------------------------
    # CABLE-Dice
    # --------------------------------------------------------
    payload_dice = load_model_checkpoint(
        model,
        best_dice_path,
    )

    dice_lesion_summary, dice_lesion_details = evaluate_lesion_loader(
        model,
        test_lesion_loader,
        save_masks_path=dice_lesion_mask,
    )

    dice_normal_summary, dice_normal_details = evaluate_normal_loader(
        model,
        test_normal_loader,
        save_masks_path=dice_normal_mask,
    )

    dice_lesion_details = dice_lesion_details.assign(
        dataset="BUS-UCLM",
        fold=int(
            fold
        ),
        train_seed=int(
            cfg.train_seed
        ),
        variant=CABLE_VARIANT,
        operating_point="dice",
    )

    dice_normal_details = dice_normal_details.assign(
        dataset="BUS-UCLM",
        fold=int(
            fold
        ),
        train_seed=int(
            cfg.train_seed
        ),
        variant=CABLE_VARIANT,
        operating_point="dice",
    )

    dice_lesion_details.to_csv(
        dice_lesion_path,
        index=False,
    )

    dice_normal_details.to_csv(
        dice_normal_path,
        index=False,
    )


    # --------------------------------------------------------
    # CABLE-Boundary
    # --------------------------------------------------------
    payload_boundary = load_model_checkpoint(
        model,
        boundary_path,
    )

    boundary_lesion_summary, boundary_lesion_details = evaluate_lesion_loader(
        model,
        test_lesion_loader,
        save_masks_path=boundary_lesion_mask,
    )

    boundary_normal_summary, boundary_normal_details = evaluate_normal_loader(
        model,
        test_normal_loader,
        save_masks_path=boundary_normal_mask,
    )

    boundary_lesion_details = boundary_lesion_details.assign(
        dataset="BUS-UCLM",
        fold=int(
            fold
        ),
        train_seed=int(
            cfg.train_seed
        ),
        variant=CABLE_VARIANT,
        operating_point="boundary",
    )

    boundary_normal_details = boundary_normal_details.assign(
        dataset="BUS-UCLM",
        fold=int(
            fold
        ),
        train_seed=int(
            cfg.train_seed
        ),
        variant=CABLE_VARIANT,
        operating_point="boundary",
    )

    boundary_lesion_details.to_csv(
        boundary_lesion_path,
        index=False,
    )

    boundary_normal_details.to_csv(
        boundary_normal_path,
        index=False,
    )


    result = {
        "dataset": "BUS-UCLM",
        "fold": int(
            fold
        ),
        "variant": CABLE_VARIANT,
        "train_seed": int(
            cfg.train_seed
        ),

        "n_train_lesion_images": int(
            len(
                split[
                    "train"
                ]
            )
        ),
        "n_val_lesion_images": int(
            len(
                split[
                    "val"
                ]
            )
        ),
        "n_test_lesion_images": int(
            len(
                split[
                    "test_lesion"
                ]
            )
        ),
        "n_test_normal_images": int(
            len(
                split[
                    "test_normal"
                ]
            )
        ),

        **selection,

        "dice_checkpoint_epoch": int(
            payload_dice[
                "epoch"
            ]
        ),

        "boundary_checkpoint_epoch": int(
            payload_boundary[
                "epoch"
            ]
        ),

        "dice_test_dice": dice_lesion_summary[
            "dice"
        ],
        "dice_test_iou": dice_lesion_summary[
            "iou"
        ],
        "dice_test_hd95": dice_lesion_summary[
            "hd95"
        ],
        "dice_test_boundary_iou": dice_lesion_summary[
            "boundary_iou"
        ],
        "dice_normal_fp_area_mean": dice_normal_summary[
            "normal_fp_area_mean"
        ],
        "dice_normal_any_lesion_rate": dice_normal_summary[
            "normal_any_lesion_rate"
        ],

        "boundary_test_dice": boundary_lesion_summary[
            "dice"
        ],
        "boundary_test_iou": boundary_lesion_summary[
            "iou"
        ],
        "boundary_test_hd95": boundary_lesion_summary[
            "hd95"
        ],
        "boundary_test_boundary_iou": boundary_lesion_summary[
            "boundary_iou"
        ],
        "boundary_normal_fp_area_mean": boundary_normal_summary[
            "normal_fp_area_mean"
        ],
        "boundary_normal_any_lesion_rate": boundary_normal_summary[
            "normal_any_lesion_rate"
        ],

        "params_M": float(
            architecture_audit.loc[
                architecture_audit.variant
                == CABLE_VARIANT,
                "params_M",
            ].iloc[
                0
            ]
        ),

        "aggregator_GMACs": float(
            complexity_lookup[
                CABLE_VARIANT
            ]
        ),

        "best_dice_checkpoint": str(
            best_dice_path
        ),

        "boundary_checkpoint": str(
            boundary_path
        ),
    }


    with open(
        result_path,
        "w",
    ) as f:
        json.dump(
            result,
            f,
            indent=2,
        )


    print(
        json.dumps(
            {
                "fold": fold,
                "Dice checkpoint epoch": result[
                    "dice_checkpoint_epoch"
                ],
                "Boundary checkpoint epoch": result[
                    "boundary_checkpoint_epoch"
                ],
                "CABLE-Dice test Dice": result[
                    "dice_test_dice"
                ],
                "CABLE-Dice test HD95": result[
                    "dice_test_hd95"
                ],
                "CABLE-Boundary test Dice": result[
                    "boundary_test_dice"
                ],
                "CABLE-Boundary test HD95": result[
                    "boundary_test_hd95"
                ],
            },
            indent=2,
        )
    )


    del (
        model,
        train_loader,
        val_loader,
        test_lesion_loader,
        test_normal_loader,
        payload_dice,
        payload_boundary,
    )

    gc.collect()

    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()


    return result


In [60]:

# ============================================================
# 24. RUN THE FIVE BUS-UCLM CABLE FOLDS
# ============================================================

uclm_cable_results = []

if RUN_BUSUCLM:
    for fold in OUTER_FOLDS:
        uclm_cable_results.append(
            run_uclm_cable_fold(
                fold
            )
        )

    uclm_cable_results_df = (
        pd.DataFrame(
            uclm_cable_results
        )
        .sort_values(
            "fold"
        )
        .reset_index(
            drop=True
        )
    )

    assert len(
        uclm_cable_results_df
    ) == 5

    uclm_cable_results_df.to_csv(
        UCLM_FINAL_ROOT
        / "CABLE_ALL_FOLD_RESULTS.csv",
        index=False,
    )

    display(
        uclm_cable_results_df[
            [
                "fold",
                "dice_checkpoint_epoch",
                "boundary_checkpoint_epoch",
                "selected_differs_from_best_dice",
                "dice_test_dice",
                "dice_test_hd95",
                "boundary_test_dice",
                "boundary_test_hd95",
                "dice_normal_fp_area_mean",
                "boundary_normal_fp_area_mean",
            ]
        ]
    )


REUSE BUS-UCLM CABLE | fold=1
REUSE BUS-UCLM CABLE | fold=2
REUSE BUS-UCLM CABLE | fold=3
REUSE BUS-UCLM CABLE | fold=4
REUSE BUS-UCLM CABLE | fold=5


,fold,dice_checkpoint_epoch,boundary_checkpoint_epoch,selected_differs_from_best_dice,dice_test_dice,dice_test_hd95,boundary_test_dice,boundary_test_hd95,dice_normal_fp_area_mean,boundary_normal_fp_area_mean
0,1,41,41,False,0.468128,65.350091,0.468128,65.350091,0.029093,0.029093
1,2,27,27,False,0.515709,67.241483,0.515709,67.241483,0.031277,0.031277
2,3,33,33,False,0.666837,48.948862,0.666837,48.948862,0.040971,0.040971
3,4,41,34,True,0.592831,56.527867,0.558833,58.457723,0.039720,0.032006
4,5,22,37,True,0.423918,60.833389,0.383575,53.005032,0.029320,0.018006



# Part C — paper-facing analysis

The main architecture table uses the **existing authoritative best-Dice baselines** and the
new **CABLE-Dice** result. CABLE-Boundary is reported separately as an operating-point
analysis and is never presented as an apples-to-apples architecture-baseline comparison.


In [61]:

# ============================================================
# 25. LOAD AUTHORITATIVE BASELINES + NEW CABLE RESULTS
# ============================================================

def load_json(
    path,
):
    with open(
        path,
        "r",
    ) as f:
        return json.load(
            f
        )


baseline_fold_rows = []


# ------------------------------------------------------------
# BUS-BRA: authoritative CLEAN checkpoint audit
# ------------------------------------------------------------
for fold in OUTER_FOLDS:
    for variant in BASELINE_VARIANTS:
        r = load_json(
            bra_baseline_dir(
                fold,
                variant,
            )
            / "result_clean.json"
        )

        baseline_fold_rows.append({
            "dataset": "BUS-BRA",
            "fold": int(
                fold
            ),
            "variant": variant,
            "operating_point": "dice",
            "test_dice": float(
                r[
                    "dice_fg_image_mean"
                ]
            ),
            "test_iou": float(
                r[
                    "iou_fg_image_mean"
                ]
            ),
            "test_hd95": float(
                r[
                    "hd95_image_mean"
                ]
            ),
            "test_boundary_iou": float(
                r[
                    "boundary_iou_image_mean"
                ]
            ),
            "normal_fp_area_mean": np.nan,
            "normal_any_lesion_rate": np.nan,
        })


# ------------------------------------------------------------
# BUS-UCLM: completed supervised patient-disjoint CV
# ------------------------------------------------------------
for fold in OUTER_FOLDS:
    for variant in BASELINE_VARIANTS:
        r = load_json(
            uclm_baseline_dir(
                fold,
                variant,
            )
            / "result.json"
        )

        baseline_fold_rows.append({
            "dataset": "BUS-UCLM",
            "fold": int(
                fold
            ),
            "variant": variant,
            "operating_point": "dice",
            "test_dice": float(
                r[
                    "test_dice"
                ]
            ),
            "test_iou": float(
                r[
                    "test_iou"
                ]
            ),
            "test_hd95": float(
                r[
                    "test_hd95"
                ]
            ),
            "test_boundary_iou": float(
                r[
                    "test_boundary_iou"
                ]
            ),
            "normal_fp_area_mean": float(
                r[
                    "normal_fp_area_mean"
                ]
            ),
            "normal_any_lesion_rate": float(
                r[
                    "normal_any_lesion_rate"
                ]
            ),
        })


baseline_fold_df = pd.DataFrame(
    baseline_fold_rows
)


# ------------------------------------------------------------
# New CABLE-Dice + CABLE-Boundary rows
# ------------------------------------------------------------
cable_fold_rows = []


if RUN_BUSBRA:
    for r in busbra_cable_results:
        for op in [
            "dice",
            "boundary",
        ]:
            cable_fold_rows.append({
                "dataset": "BUS-BRA",
                "fold": int(
                    r[
                        "fold"
                    ]
                ),
                "variant": CABLE_VARIANT,
                "operating_point": op,
                "test_dice": float(
                    r[
                        f"{op}_test_dice"
                    ]
                ),
                "test_iou": float(
                    r[
                        f"{op}_test_iou"
                    ]
                ),
                "test_hd95": float(
                    r[
                        f"{op}_test_hd95"
                    ]
                ),
                "test_boundary_iou": float(
                    r[
                        f"{op}_test_boundary_iou"
                    ]
                ),
                "normal_fp_area_mean": np.nan,
                "normal_any_lesion_rate": np.nan,
            })


if RUN_BUSUCLM:
    for r in uclm_cable_results:
        for op in [
            "dice",
            "boundary",
        ]:
            cable_fold_rows.append({
                "dataset": "BUS-UCLM",
                "fold": int(
                    r[
                        "fold"
                    ]
                ),
                "variant": CABLE_VARIANT,
                "operating_point": op,
                "test_dice": float(
                    r[
                        f"{op}_test_dice"
                    ]
                ),
                "test_iou": float(
                    r[
                        f"{op}_test_iou"
                    ]
                ),
                "test_hd95": float(
                    r[
                        f"{op}_test_hd95"
                    ]
                ),
                "test_boundary_iou": float(
                    r[
                        f"{op}_test_boundary_iou"
                    ]
                ),
                "normal_fp_area_mean": float(
                    r[
                        f"{op}_normal_fp_area_mean"
                    ]
                ),
                "normal_any_lesion_rate": float(
                    r[
                        f"{op}_normal_any_lesion_rate"
                    ]
                ),
            })


cable_fold_df = pd.DataFrame(
    cable_fold_rows
)


all_fold_df = pd.concat(
    [
        baseline_fold_df,
        cable_fold_df,
    ],
    ignore_index=True,
)


all_fold_df.to_csv(
    FINAL_ROOT
    / "ALL_METHODS_ALL_OPERATING_POINTS_FOLD_RESULTS.csv",
    index=False,
)


# Main fair architecture comparison:
main_fold_df = all_fold_df[
    all_fold_df[
        "operating_point"
    ]
    == "dice"
].copy()


assert (
    main_fold_df.groupby(
        [
            "dataset",
            "variant",
        ]
    )[
        "fold"
    ]
    .nunique()
    == 5
).all()


display(
    main_fold_df.sort_values(
        [
            "dataset",
            "fold",
            "variant",
        ]
    )
)


,dataset,fold,variant,operating_point,test_dice,test_iou,test_hd95,test_boundary_iou,normal_fp_area_mean,normal_any_lesion_rate
20,BUS-BRA,1,cable_s1,dice,0.855043,0.768337,17.635295,0.311087,NaN,NaN
1,BUS-BRA,1,mhtm_hybrid_final,dice,0.852776,0.765015,18.771803,0.301415,NaN,NaN
0,BUS-BRA,1,swift_lgl,dice,0.858946,0.772448,16.728889,0.309644,NaN,NaN
22,BUS-BRA,2,cable_s1,dice,0.838175,0.748393,20.051138,0.298238,NaN,NaN
3,BUS-BRA,2,mhtm_hybrid_final,dice,0.831522,0.738712,21.767122,0.287388,NaN,NaN
2,BUS-BRA,2,swift_lgl,dice,0.839043,0.751184,18.371030,0.304459,NaN,NaN
24,BUS-BRA,3,cable_s1,dice,0.856073,0.768018,17.445600,0.306027,NaN,NaN
5,BUS-BRA,3,mhtm_hybrid_final,dice,0.852770,0.764623,18.857680,0.304917,NaN,NaN
4,BUS-BRA,3,swift_lgl,dice,0.852965,0.763085,18.072275,0.304560,NaN,NaN
26,BUS-BRA,4,cable_s1,dice,0.845700,0.757963,20.106434,0.292923,NaN,NaN


In [62]:

# ============================================================
# 26. MAIN ARCHITECTURE TABLE + CABLE OPERATING-POINT TABLE
# ============================================================

def summarize_folds(
    df,
):
    rows = []

    for (
        dataset,
        variant,
        op,
    ), g in df.groupby(
        [
            "dataset",
            "variant",
            "operating_point",
        ]
    ):
        rows.append({
            "dataset": dataset,
            "variant": variant,
            "operating_point": op,
            "folds": int(
                g.fold.nunique()
            ),
            "dice_mean": float(
                g.test_dice.mean()
            ),
            "dice_sd": float(
                g.test_dice.std(
                    ddof=1
                )
            ),
            "iou_mean": float(
                g.test_iou.mean()
            ),
            "iou_sd": float(
                g.test_iou.std(
                    ddof=1
                )
            ),
            "hd95_mean": float(
                g.test_hd95.mean()
            ),
            "hd95_sd": float(
                g.test_hd95.std(
                    ddof=1
                )
            ),
            "boundary_iou_mean": float(
                g.test_boundary_iou.mean()
            ),
            "boundary_iou_sd": float(
                g.test_boundary_iou.std(
                    ddof=1
                )
            ),
            "normal_fp_area_mean": (
                float(
                    g.normal_fp_area_mean.mean()
                )
                if g.normal_fp_area_mean.notna().any()
                else np.nan
            ),
            "normal_any_lesion_rate_mean": (
                float(
                    g.normal_any_lesion_rate.mean()
                )
                if g.normal_any_lesion_rate.notna().any()
                else np.nan
            ),
        })

    return pd.DataFrame(
        rows
    )


main_summary = summarize_folds(
    main_fold_df
)

cable_op_summary = summarize_folds(
    cable_fold_df
)


eff = efficiency_df.set_index(
    "variant"
)

for table in [
    main_summary,
    cable_op_summary,
]:
    for col in [
        "params_M",
        "aggregator_GMACs",
        "latency_ms_median",
        "peak_inference_memory_mb",
    ]:
        table[
            col
        ] = table.variant.map(
            eff[
                col
            ]
        )


print(
    "MAIN FAIR ARCHITECTURE COMPARISON "
    "(all methods selected by validation Dice)"
)

display(
    main_summary.sort_values(
        [
            "dataset",
            "variant",
        ]
    )
)


print(
    "\nCABLE OPERATING-POINT ANALYSIS"
)

display(
    cable_op_summary.sort_values(
        [
            "dataset",
            "operating_point",
        ]
    )
)


main_summary.to_csv(
    FINAL_ROOT
    / "MAIN_ARCHITECTURE_TABLE.csv",
    index=False,
)

cable_op_summary.to_csv(
    FINAL_ROOT
    / "CABLE_OPERATING_POINT_TABLE.csv",
    index=False,
)


MAIN FAIR ARCHITECTURE COMPARISON (all methods selected by validation Dice)


,dataset,variant,operating_point,folds,dice_mean,dice_sd,iou_mean,iou_sd,hd95_mean,hd95_sd,boundary_iou_mean,boundary_iou_sd,normal_fp_area_mean,normal_any_lesion_rate_mean,params_M,aggregator_GMACs,latency_ms_median,peak_inference_memory_mb
0,BUS-BRA,cable_s1,dice,5,0.848492,0.007343,0.760241,0.008286,18.951716,1.310203,0.300883,0.007474,NaN,NaN,4.487235,0.231195,12.685904,47.681641
1,BUS-BRA,mhtm_hybrid_final,dice,5,0.846394,0.009110,0.757590,0.011245,19.686632,1.228710,0.297414,0.006659,NaN,NaN,4.481858,0.225690,13.944832,47.658691
2,BUS-BRA,swift_lgl,dice,5,0.853465,0.011214,0.767448,0.014306,17.510341,0.923936,0.307290,0.007669,NaN,NaN,4.485282,1.312145,13.921840,47.674805
3,BUS-UCLM,cable_s1,dice,5,0.533485,0.097336,0.411671,0.094117,59.780338,7.343351,0.091841,0.029980,0.034076,0.823262,4.487235,0.231195,12.685904,47.681641
4,BUS-UCLM,mhtm_hybrid_final,dice,5,0.604424,0.061878,0.480628,0.060332,56.161526,6.117363,0.105596,0.020240,0.034549,0.867587,4.481858,0.225690,13.944832,47.658691
5,BUS-UCLM,swift_lgl,dice,5,0.544929,0.089801,0.421944,0.086417,60.062973,20.689885,0.087502,0.029274,0.037237,0.856257,4.485282,1.312145,13.921840,47.674805



CABLE OPERATING-POINT ANALYSIS


,dataset,variant,operating_point,folds,dice_mean,dice_sd,iou_mean,iou_sd,hd95_mean,hd95_sd,boundary_iou_mean,boundary_iou_sd,normal_fp_area_mean,normal_any_lesion_rate_mean,params_M,aggregator_GMACs,latency_ms_median,peak_inference_memory_mb
0,BUS-BRA,cable_s1,boundary,5,0.847292,0.009161,0.758957,0.010411,18.287643,1.612403,0.301730,0.009707,NaN,NaN,4.487235,0.231195,12.685904,47.681641
1,BUS-BRA,cable_s1,dice,5,0.848492,0.007343,0.760241,0.008286,18.951716,1.310203,0.300883,0.007474,NaN,NaN,4.487235,0.231195,12.685904,47.681641
2,BUS-UCLM,cable_s1,boundary,5,0.518616,0.105333,0.400784,0.098193,58.600638,7.821562,0.089907,0.029799,0.030271,0.827492,4.487235,0.231195,12.685904,47.681641
3,BUS-UCLM,cable_s1,dice,5,0.533485,0.097336,0.411671,0.094117,59.780338,7.343351,0.091841,0.029980,0.034076,0.823262,4.487235,0.231195,12.685904,47.681641


In [63]:

# ============================================================
# 27. NORMALIZE + AUDIT ALL OOF PER-IMAGE RESULTS
# ============================================================

def normalize_bra_baseline_details(
    df,
):
    out = pd.DataFrame({
        "id": df[
            "id"
        ].astype(
            str
        ),
        "patient": df[
            "case"
        ].astype(
            str
        ),
        "pathology": df[
            "pathology"
        ].astype(
            str
        ),
        "dice": df[
            "dice_fg"
        ].astype(
            float
        ),
        "iou": df[
            "iou_fg"
        ].astype(
            float
        ),
        "hd95": df[
            "hd95"
        ].astype(
            float
        ),
        "boundary_iou": df[
            "boundary_iou"
        ].astype(
            float
        ),
        "gt_area_fraction": df[
            "target_area_fraction"
        ].astype(
            float
        ),
        "pred_area_fraction": df[
            "pred_area_fraction"
        ].astype(
            float
        ),
        "fold": df[
            "fold"
        ].astype(
            int
        ),
        "variant": df[
            "variant"
        ].astype(
            str
        ),
    })

    out[
        "operating_point"
    ] = "dice"

    out[
        "dataset"
    ] = "BUS-BRA"

    return out


def normalize_uclm_baseline_details(
    df,
):
    out = df[
        [
            "id",
            "patient",
            "pathology",
            "dice",
            "iou",
            "hd95",
            "boundary_iou",
            "gt_area_fraction",
            "pred_area_fraction",
            "fold",
            "variant",
        ]
    ].copy()

    out[
        "id"
    ] = out[
        "id"
    ].astype(
        str
    )

    out[
        "patient"
    ] = out[
        "patient"
    ].astype(
        str
    )

    out[
        "operating_point"
    ] = "dice"

    out[
        "dataset"
    ] = "BUS-UCLM"

    return out


oof_frames = []
normal_frames = []


# Existing BUS-BRA clean audit.
for fold in OUTER_FOLDS:
    for variant in BASELINE_VARIANTS:
        d = pd.read_csv(
            bra_baseline_dir(
                fold,
                variant,
            )
            / "per_image_test_clean.csv"
        )

        oof_frames.append(
            normalize_bra_baseline_details(
                d
            )
        )


# Existing BUS-UCLM supervised baselines.
for fold in OUTER_FOLDS:
    for variant in BASELINE_VARIANTS:
        d = pd.read_csv(
            uclm_baseline_dir(
                fold,
                variant,
            )
            / "per_image_test_lesion.csv"
        )

        oof_frames.append(
            normalize_uclm_baseline_details(
                d
            )
        )

        n = pd.read_csv(
            uclm_baseline_dir(
                fold,
                variant,
            )
            / "per_image_test_normal.csv"
        )

        n[
            "dataset"
        ] = "BUS-UCLM"

        n[
            "operating_point"
        ] = "dice"

        normal_frames.append(
            n
        )


# New CABLE operating points.
if RUN_BUSBRA:
    for fold in OUTER_FOLDS:
        out_dir = busbra_cable_run_dir(
            fold
        )

        for op in [
            "dice",
            "boundary",
        ]:
            d = pd.read_csv(
                out_dir
                / f"per_image_test_{op}.csv"
            )

            oof_frames.append(
                d
            )


if RUN_BUSUCLM:
    for fold in OUTER_FOLDS:
        out_dir = uclm_cable_run_dir(
            fold
        )

        for op in [
            "dice",
            "boundary",
        ]:
            d = pd.read_csv(
                out_dir
                / f"per_image_test_lesion_{op}.csv"
            )

            n = pd.read_csv(
                out_dir
                / f"per_image_test_normal_{op}.csv"
            )

            oof_frames.append(
                d
            )

            normal_frames.append(
                n
            )


all_oof = pd.concat(
    oof_frames,
    ignore_index=True,
)

all_normal_oof = pd.concat(
    normal_frames,
    ignore_index=True,
)


# ------------------------------------------------------------
# Hard OOF coverage gates
# ------------------------------------------------------------

expected_lesion_rows = {
    (
        "BUS-BRA",
        "swift_lgl",
        "dice",
    ): 1875,
    (
        "BUS-BRA",
        "mhtm_hybrid_final",
        "dice",
    ): 1875,
    (
        "BUS-BRA",
        "cable_s1",
        "dice",
    ): 1875,
    (
        "BUS-BRA",
        "cable_s1",
        "boundary",
    ): 1875,

    (
        "BUS-UCLM",
        "swift_lgl",
        "dice",
    ): 260,
    (
        "BUS-UCLM",
        "mhtm_hybrid_final",
        "dice",
    ): 260,
    (
        "BUS-UCLM",
        "cable_s1",
        "dice",
    ): 260,
    (
        "BUS-UCLM",
        "cable_s1",
        "boundary",
    ): 260,
}


for key, expected in expected_lesion_rows.items():
    dataset, variant, op = key

    g = all_oof[
        (
            all_oof.dataset
            == dataset
        )
        & (
            all_oof.variant
            == variant
        )
        & (
            all_oof.operating_point
            == op
        )
    ]

    assert len(
        g
    ) == expected, (
        key,
        len(
            g
        ),
        expected,
    )

    assert g[
        "id"
    ].nunique() == expected


for op in [
    "dice",
    "boundary",
]:
    g = all_normal_oof[
        (
            all_normal_oof.variant
            == "cable_s1"
        )
        & (
            all_normal_oof.operating_point
            == op
        )
    ]

    assert len(
        g
    ) == 410

    assert g[
        "id"
    ].nunique() == 410


print(
    "PASS: authoritative baselines + both CABLE operating points "
    "have complete OOF coverage."
)


all_oof.to_csv(
    FINAL_ROOT
    / "ALL_OOF_LESION_RESULTS.csv",
    index=False,
)

all_normal_oof.to_csv(
    FINAL_ROOT
    / "ALL_OOF_NORMAL_RESULTS.csv",
    index=False,
)


PASS: authoritative baselines + both CABLE operating points have complete OOF coverage.


In [64]:

# ============================================================
# 28. PATIENT-CLUSTERED PAIRED BOOTSTRAP
# ============================================================

def patient_bootstrap_delta(
    reference_df,
    comparison_df,
    metric,
    n_boot=5000,
    seed=2026,
):
    a = (
        reference_df[
            [
                "id",
                "patient",
                metric,
            ]
        ]
        .rename(
            columns={
                metric: "reference"
            }
        )
    )

    b = (
        comparison_df[
            [
                "id",
                "patient",
                metric,
            ]
        ]
        .rename(
            columns={
                metric: "comparison"
            }
        )
    )

    paired = a.merge(
        b,
        on=[
            "id",
            "patient",
        ],
        validate="one_to_one",
    )

    paired[
        "delta"
    ] = (
        paired.comparison
        - paired.reference
    )

    patient_delta = (
        paired.groupby(
            "patient"
        )[
            "delta"
        ]
        .mean()
    )

    values = patient_delta.to_numpy(
        dtype=float
    )

    rng = np.random.default_rng(
        seed
    )

    boots = np.empty(
        n_boot,
        dtype=float,
    )

    for i in range(
        n_boot
    ):
        idx = rng.integers(
            0,
            len(
                values
            ),
            size=len(
                values
            ),
        )

        boots[
            i
        ] = values[
            idx
        ].mean()

    lo, hi = np.percentile(
        boots,
        [
            2.5,
            97.5,
        ],
    )

    return {
        "metric": metric,
        "delta_comparison_minus_reference": float(
            values.mean()
        ),
        "ci95_low": float(
            lo
        ),
        "ci95_high": float(
            hi
        ),
        "n_patients": int(
            len(
                values
            )
        ),
        "n_images": int(
            len(
                paired
            )
        ),
    }


bootstrap_rows = []


# ------------------------------------------------------------
# Main fair architecture comparison:
# CABLE-Dice vs each best-Dice baseline.
# ------------------------------------------------------------
for dataset in [
    "BUS-BRA",
    "BUS-UCLM",
]:
    cable_dice = all_oof[
        (
            all_oof.dataset
            == dataset
        )
        & (
            all_oof.variant
            == CABLE_VARIANT
        )
        & (
            all_oof.operating_point
            == "dice"
        )
    ]

    for ref_variant in BASELINE_VARIANTS:
        ref = all_oof[
            (
                all_oof.dataset
                == dataset
            )
            & (
                all_oof.variant
                == ref_variant
            )
            & (
                all_oof.operating_point
                == "dice"
            )
        ]

        for metric in [
            "dice",
            "iou",
            "hd95",
            "boundary_iou",
        ]:
            row = patient_bootstrap_delta(
                ref,
                cable_dice,
                metric,
            )

            row.update({
                "dataset": dataset,
                "analysis": "main_architecture",
                "reference": ref_variant,
                "comparison": "cable_s1_dice",
            })

            bootstrap_rows.append(
                row
            )


# ------------------------------------------------------------
# Secondary operating-point comparison:
# CABLE-Boundary vs CABLE-Dice.
# ------------------------------------------------------------
for dataset in [
    "BUS-BRA",
    "BUS-UCLM",
]:
    cable_dice = all_oof[
        (
            all_oof.dataset
            == dataset
        )
        & (
            all_oof.variant
            == CABLE_VARIANT
        )
        & (
            all_oof.operating_point
            == "dice"
        )
    ]

    cable_boundary = all_oof[
        (
            all_oof.dataset
            == dataset
        )
        & (
            all_oof.variant
            == CABLE_VARIANT
        )
        & (
            all_oof.operating_point
            == "boundary"
        )
    ]

    for metric in [
        "dice",
        "iou",
        "hd95",
        "boundary_iou",
    ]:
        row = patient_bootstrap_delta(
            cable_dice,
            cable_boundary,
            metric,
        )

        row.update({
            "dataset": dataset,
            "analysis": "cable_operating_point",
            "reference": "cable_s1_dice",
            "comparison": "cable_s1_boundary",
        })

        bootstrap_rows.append(
            row
        )


bootstrap_df = pd.DataFrame(
    bootstrap_rows
)

display(
    bootstrap_df
)

bootstrap_df.to_csv(
    FINAL_ROOT
    / "PATIENT_CLUSTERED_BOOTSTRAP.csv",
    index=False,
)


,metric,delta_comparison_minus_reference,ci95_low,ci95_high,n_patients,n_images,dataset,analysis,reference,comparison
0,dice,-0.005278,-0.009500,-0.000810,1064,1875,BUS-BRA,main_architecture,swift_lgl,cable_s1_dice
1,iou,-0.007452,-0.012537,-0.002155,1064,1875,BUS-BRA,main_architecture,swift_lgl,cable_s1_dice
2,hd95,1.579305,0.751226,2.423623,1064,1875,BUS-BRA,main_architecture,swift_lgl,cable_s1_dice
3,boundary_iou,-0.005991,-0.009988,-0.001968,1064,1875,BUS-BRA,main_architecture,swift_lgl,cable_s1_dice
4,dice,0.002313,-0.001752,0.006413,1064,1875,BUS-BRA,main_architecture,mhtm_hybrid_final,cable_s1_dice
5,iou,0.002892,-0.001974,0.007830,1064,1875,BUS-BRA,main_architecture,mhtm_hybrid_final,cable_s1_dice
6,hd95,-0.650446,-1.499459,0.173298,1064,1875,BUS-BRA,main_architecture,mhtm_hybrid_final,cable_s1_dice
7,boundary_iou,0.004345,0.000462,0.008125,1064,1875,BUS-BRA,main_architecture,mhtm_hybrid_final,cable_s1_dice
8,dice,-0.022527,-0.068239,0.023928,35,260,BUS-UCLM,main_architecture,swift_lgl,cable_s1_dice
9,iou,-0.024233,-0.065755,0.017831,35,260,BUS-UCLM,main_architecture,swift_lgl,cable_s1_dice


In [65]:

# ============================================================
# 29. CABLE CHECKPOINT BEHAVIOR + SUBGROUP ANALYSIS
# ============================================================

selection_rows = []

if RUN_BUSBRA:
    for r in busbra_cable_results:
        selection_rows.append({
            "dataset": "BUS-BRA",
            "fold": int(
                r[
                    "fold"
                ]
            ),
            "best_dice_epoch": int(
                r[
                    "dice_checkpoint_epoch"
                ]
            ),
            "boundary_epoch": int(
                r[
                    "boundary_checkpoint_epoch"
                ]
            ),
            "changed": bool(
                r[
                    "selected_differs_from_best_dice"
                ]
            ),
            "val_max_dice": float(
                r[
                    "max_val_dice"
                ]
            ),
            "boundary_selected_val_dice": float(
                r[
                    "selected_val_dice"
                ]
            ),
            "boundary_selected_val_hd95": float(
                r[
                    "selected_val_hd95"
                ]
            ),
        })


if RUN_BUSUCLM:
    for r in uclm_cable_results:
        selection_rows.append({
            "dataset": "BUS-UCLM",
            "fold": int(
                r[
                    "fold"
                ]
            ),
            "best_dice_epoch": int(
                r[
                    "dice_checkpoint_epoch"
                ]
            ),
            "boundary_epoch": int(
                r[
                    "boundary_checkpoint_epoch"
                ]
            ),
            "changed": bool(
                r[
                    "selected_differs_from_best_dice"
                ]
            ),
            "val_max_dice": float(
                r[
                    "max_val_dice"
                ]
            ),
            "boundary_selected_val_dice": float(
                r[
                    "selected_val_dice"
                ]
            ),
            "boundary_selected_val_hd95": float(
                r[
                    "selected_val_hd95"
                ]
            ),
        })


selection_df = pd.DataFrame(
    selection_rows
)

display(
    selection_df
)

print(
    "\nFraction of folds where the boundary-aware selector "
    "chooses a different epoch:"
)

display(
    selection_df.groupby(
        "dataset"
    )[
        "changed"
    ].mean()
)


selection_df.to_csv(
    FINAL_ROOT
    / "CABLE_CHECKPOINT_SELECTION_AUDIT.csv",
    index=False,
)


# ------------------------------------------------------------
# Pathology subgroup
# ------------------------------------------------------------
pathology_summary = (
    all_oof.groupby(
        [
            "dataset",
            "variant",
            "operating_point",
            "pathology",
        ],
        as_index=False,
    )
    .agg(
        n_images=("id", "nunique"),
        n_patients=("patient", "nunique"),
        dice=("dice", "mean"),
        iou=("iou", "mean"),
        hd95=("hd95", "mean"),
        boundary_iou=("boundary_iou", "mean"),
    )
)

display(
    pathology_summary
)

pathology_summary.to_csv(
    FINAL_ROOT
    / "PATHOLOGY_SUMMARY.csv",
    index=False,
)


# ------------------------------------------------------------
# Lesion-size sensitivity
# ------------------------------------------------------------
size_frames = []

for dataset in [
    "BUS-BRA",
    "BUS-UCLM",
]:
    ref = all_oof[
        (
            all_oof.dataset
            == dataset
        )
        & (
            all_oof.variant
            == "swift_lgl"
        )
        & (
            all_oof.operating_point
            == "dice"
        )
    ][
        [
            "id",
            "gt_area_fraction",
        ]
    ].drop_duplicates(
        "id"
    )

    q25, q75 = np.quantile(
        ref[
            "gt_area_fraction"
        ],
        [
            0.25,
            0.75,
        ],
    )

    tmp = all_oof[
        all_oof.dataset
        == dataset
    ].copy()

    tmp[
        "size_group"
    ] = np.where(
        tmp[
            "gt_area_fraction"
        ]
        <= q25,
        "small_Q1",
        np.where(
            tmp[
                "gt_area_fraction"
            ]
            >= q75,
            "large_Q4",
            "middle_Q2_Q3",
        ),
    )

    s = (
        tmp.groupby(
            [
                "dataset",
                "variant",
                "operating_point",
                "size_group",
            ],
            as_index=False,
        )
        .agg(
            n_images=("id", "nunique"),
            dice=("dice", "mean"),
            iou=("iou", "mean"),
            hd95=("hd95", "mean"),
            boundary_iou=("boundary_iou", "mean"),
        )
    )

    s[
        "q25_area_fraction"
    ] = q25

    s[
        "q75_area_fraction"
    ] = q75

    size_frames.append(
        s
    )


size_summary = pd.concat(
    size_frames,
    ignore_index=True,
)

display(
    size_summary
)

size_summary.to_csv(
    FINAL_ROOT
    / "LESION_SIZE_SUMMARY.csv",
    index=False,
)


,dataset,fold,best_dice_epoch,boundary_epoch,changed,val_max_dice,boundary_selected_val_dice,boundary_selected_val_hd95
0,BUS-BRA,1,45,48,True,0.850690,0.848090,18.002196
1,BUS-BRA,2,39,49,True,0.862048,0.860732,16.505384
2,BUS-BRA,3,50,50,False,0.846514,0.846514,19.403443
3,BUS-BRA,4,32,30,True,0.855565,0.850709,16.584896
4,BUS-BRA,5,50,48,True,0.846939,0.845077,17.445541
5,BUS-UCLM,1,41,41,False,0.538294,0.538294,76.555743
6,BUS-UCLM,2,27,27,False,0.470408,0.470408,69.409920
7,BUS-UCLM,3,33,33,False,0.579934,0.579934,59.319464
8,BUS-UCLM,4,41,34,True,0.489127,0.488309,84.549874
9,BUS-UCLM,5,22,37,True,0.558952,0.556466,45.180606



Fraction of folds where the boundary-aware selector chooses a different epoch:


,changed
dataset,
BUS-BRA,0.8
BUS-UCLM,0.4


,dataset,variant,operating_point,pathology,n_images,n_patients,dice,iou,hd95,boundary_iou
0,BUS-BRA,cable_s1,boundary,benign,1268,722,0.860811,0.778026,15.815559,0.338016
1,BUS-BRA,cable_s1,boundary,malignant,607,342,0.818947,0.718990,23.505104,0.225994
2,BUS-BRA,cable_s1,dice,benign,1268,722,0.860387,0.777132,16.571032,0.335083
3,BUS-BRA,cable_s1,dice,malignant,607,342,0.823406,0.724673,23.951620,0.229408
4,BUS-BRA,mhtm_hybrid_final,dice,benign,1268,722,0.857787,0.774255,17.625595,0.331315
5,BUS-BRA,mhtm_hybrid_final,dice,malignant,607,342,0.822371,0.722492,24.030121,0.226367
6,BUS-BRA,swift_lgl,dice,benign,1268,722,0.865963,0.786051,15.061526,0.344294
7,BUS-BRA,swift_lgl,dice,malignant,607,342,0.827430,0.728778,22.605401,0.230285
8,BUS-UCLM,cable_s1,boundary,benign,170,23,0.546315,0.428814,59.586241,0.110967
9,BUS-UCLM,cable_s1,boundary,malignant,90,17,0.514182,0.394774,57.832147,0.060112


,dataset,variant,operating_point,size_group,n_images,dice,iou,hd95,boundary_iou,q25_area_fraction,q75_area_fraction
0,BUS-BRA,cable_s1,boundary,large_Q4,469,0.844498,0.759294,24.350727,0.231332,0.032997,0.118134
1,BUS-BRA,cable_s1,boundary,middle_Q2_Q3,937,0.857573,0.770053,16.552471,0.302992,0.032997,0.118134
2,BUS-BRA,cable_s1,boundary,small_Q1,469,0.829408,0.736281,15.760278,0.369689,0.032997,0.118134
3,BUS-BRA,cable_s1,dice,large_Q4,469,0.853993,0.770391,23.399147,0.237688,0.032997,0.118134
4,BUS-BRA,cable_s1,dice,middle_Q2_Q3,937,0.858701,0.771281,17.430880,0.303510,0.032997,0.118134
5,BUS-BRA,cable_s1,dice,small_Q1,469,0.822287,0.727669,17.577327,0.358788,0.032997,0.118134
6,BUS-BRA,mhtm_hybrid_final,dice,large_Q4,469,0.855781,0.770231,23.631339,0.229663,0.032997,0.118134
7,BUS-BRA,mhtm_hybrid_final,dice,middle_Q2_Q3,937,0.858397,0.770762,18.179976,0.298947,0.032997,0.118134
8,BUS-BRA,mhtm_hybrid_final,dice,small_Q1,469,0.812737,0.718262,18.801286,0.361806,0.032997,0.118134
9,BUS-BRA,swift_lgl,dice,large_Q4,469,0.860672,0.779206,21.839693,0.239839,0.032997,0.118134


In [66]:

# ============================================================
# 30. LATEX-READY MAIN + OPERATING-POINT TABLES
# ============================================================

LABELS = {
    "swift_lgl": "SwiftMSeg-LGL",
    "mhtm_hybrid_final": "MHTM-Local",
    "cable_s1": "CABLE",
}


def mean_sd(
    mean,
    sd,
    digits=4,
):
    return (
        f"{mean:.{digits}f} "
        f"$\\pm$ "
        f"{sd:.{digits}f}"
    )


main_latex_rows = []

for row in main_summary.sort_values(
    [
        "dataset",
        "variant",
    ]
).itertuples(
    index=False
):
    main_latex_rows.append({
        "Dataset": row.dataset,
        "Method": LABELS[
            row.variant
        ],
        "Dice": mean_sd(
            row.dice_mean,
            row.dice_sd,
        ),
        "IoU": mean_sd(
            row.iou_mean,
            row.iou_sd,
        ),
        "HD95": mean_sd(
            row.hd95_mean,
            row.hd95_sd,
            digits=2,
        ),
        "B-IoU": mean_sd(
            row.boundary_iou_mean,
            row.boundary_iou_sd,
        ),
        "Params (M)": f"{row.params_M:.3f}",
        "Agg. GMACs": f"{row.aggregator_GMACs:.3f}",
        "Latency (ms)": f"{row.latency_ms_median:.2f}",
    })


main_latex_df = pd.DataFrame(
    main_latex_rows
)

print(
    "MAIN FAIR ARCHITECTURE TABLE"
)

display(
    main_latex_df
)

main_latex = main_latex_df.to_latex(
    index=False,
    escape=False,
)

print(
    main_latex
)

with open(
    FINAL_ROOT
    / "MAIN_ARCHITECTURE_TABLE.tex",
    "w",
) as f:
    f.write(
        main_latex
    )


op_latex_rows = []

for row in cable_op_summary.sort_values(
    [
        "dataset",
        "operating_point",
    ]
).itertuples(
    index=False
):
    op_label = (
        "CABLE-Dice"
        if row.operating_point
        == "dice"
        else "CABLE-Boundary"
    )

    op_latex_rows.append({
        "Dataset": row.dataset,
        "Operating point": op_label,
        "Dice": mean_sd(
            row.dice_mean,
            row.dice_sd,
        ),
        "IoU": mean_sd(
            row.iou_mean,
            row.iou_sd,
        ),
        "HD95": mean_sd(
            row.hd95_mean,
            row.hd95_sd,
            digits=2,
        ),
        "B-IoU": mean_sd(
            row.boundary_iou_mean,
            row.boundary_iou_sd,
        ),
    })


op_latex_df = pd.DataFrame(
    op_latex_rows
)

print(
    "\nCABLE OPERATING-POINT TABLE"
)

display(
    op_latex_df
)

op_latex = op_latex_df.to_latex(
    index=False,
    escape=False,
)

print(
    op_latex
)

with open(
    FINAL_ROOT
    / "CABLE_OPERATING_POINT_TABLE.tex",
    "w",
) as f:
    f.write(
        op_latex
    )


MAIN FAIR ARCHITECTURE TABLE


,Dataset,Method,Dice,IoU,HD95,B-IoU,Params (M),Agg. GMACs,Latency (ms)
0,BUS-BRA,CABLE,0.8485 $\pm$ 0.0073,0.7602 $\pm$ 0.0083,18.95 $\pm$ 1.31,0.3009 $\pm$ 0.0075,4.487,0.231,12.69
1,BUS-BRA,MHTM-Local,0.8464 $\pm$ 0.0091,0.7576 $\pm$ 0.0112,19.69 $\pm$ 1.23,0.2974 $\pm$ 0.0067,4.482,0.226,13.94
2,BUS-BRA,SwiftMSeg-LGL,0.8535 $\pm$ 0.0112,0.7674 $\pm$ 0.0143,17.51 $\pm$ 0.92,0.3073 $\pm$ 0.0077,4.485,1.312,13.92
3,BUS-UCLM,CABLE,0.5335 $\pm$ 0.0973,0.4117 $\pm$ 0.0941,59.78 $\pm$ 7.34,0.0918 $\pm$ 0.0300,4.487,0.231,12.69
4,BUS-UCLM,MHTM-Local,0.6044 $\pm$ 0.0619,0.4806 $\pm$ 0.0603,56.16 $\pm$ 6.12,0.1056 $\pm$ 0.0202,4.482,0.226,13.94
5,BUS-UCLM,SwiftMSeg-LGL,0.5449 $\pm$ 0.0898,0.4219 $\pm$ 0.0864,60.06 $\pm$ 20.69,0.0875 $\pm$ 0.0293,4.485,1.312,13.92


\begin{tabular}{lllllllll}
\toprule
Dataset & Method & Dice & IoU & HD95 & B-IoU & Params (M) & Agg. GMACs & Latency (ms) \\
\midrule
BUS-BRA & CABLE & 0.8485 $\pm$ 0.0073 & 0.7602 $\pm$ 0.0083 & 18.95 $\pm$ 1.31 & 0.3009 $\pm$ 0.0075 & 4.487 & 0.231 & 12.69 \\
BUS-BRA & MHTM-Local & 0.8464 $\pm$ 0.0091 & 0.7576 $\pm$ 0.0112 & 19.69 $\pm$ 1.23 & 0.2974 $\pm$ 0.0067 & 4.482 & 0.226 & 13.94 \\
BUS-BRA & SwiftMSeg-LGL & 0.8535 $\pm$ 0.0112 & 0.7674 $\pm$ 0.0143 & 17.51 $\pm$ 0.92 & 0.3073 $\pm$ 0.0077 & 4.485 & 1.312 & 13.92 \\
BUS-UCLM & CABLE & 0.5335 $\pm$ 0.0973 & 0.4117 $\pm$ 0.0941 & 59.78 $\pm$ 7.34 & 0.0918 $\pm$ 0.0300 & 4.487 & 0.231 & 12.69 \\
BUS-UCLM & MHTM-Local & 0.6044 $\pm$ 0.0619 & 0.4806 $\pm$ 0.0603 & 56.16 $\pm$ 6.12 & 0.1056 $\pm$ 0.0202 & 4.482 & 0.226 & 13.94 \\
BUS-UCLM & SwiftMSeg-LGL & 0.5449 $\pm$ 0.0898 & 0.4219 $\pm$ 0.0864 & 60.06 $\pm$ 20.69 & 0.0875 $\pm$ 0.0293 & 4.485 & 1.312 & 13.92 \\
\bottomrule
\end{tabular}


CABLE OPERATING-POINT TABLE


,Dataset,Operating point,Dice,IoU,HD95,B-IoU
0,BUS-BRA,CABLE-Boundary,0.8473 $\pm$ 0.0092,0.7590 $\pm$ 0.0104,18.29 $\pm$ 1.61,0.3017 $\pm$ 0.0097
1,BUS-BRA,CABLE-Dice,0.8485 $\pm$ 0.0073,0.7602 $\pm$ 0.0083,18.95 $\pm$ 1.31,0.3009 $\pm$ 0.0075
2,BUS-UCLM,CABLE-Boundary,0.5186 $\pm$ 0.1053,0.4008 $\pm$ 0.0982,58.60 $\pm$ 7.82,0.0899 $\pm$ 0.0298
3,BUS-UCLM,CABLE-Dice,0.5335 $\pm$ 0.0973,0.4117 $\pm$ 0.0941,59.78 $\pm$ 7.34,0.0918 $\pm$ 0.0300


\begin{tabular}{llllll}
\toprule
Dataset & Operating point & Dice & IoU & HD95 & B-IoU \\
\midrule
BUS-BRA & CABLE-Boundary & 0.8473 $\pm$ 0.0092 & 0.7590 $\pm$ 0.0104 & 18.29 $\pm$ 1.61 & 0.3017 $\pm$ 0.0097 \\
BUS-BRA & CABLE-Dice & 0.8485 $\pm$ 0.0073 & 0.7602 $\pm$ 0.0083 & 18.95 $\pm$ 1.31 & 0.3009 $\pm$ 0.0075 \\
BUS-UCLM & CABLE-Boundary & 0.5186 $\pm$ 0.1053 & 0.4008 $\pm$ 0.0982 & 58.60 $\pm$ 7.82 & 0.0899 $\pm$ 0.0298 \\
BUS-UCLM & CABLE-Dice & 0.5335 $\pm$ 0.0973 & 0.4117 $\pm$ 0.0941 & 59.78 $\pm$ 7.34 & 0.0918 $\pm$ 0.0300 \\
\bottomrule
\end{tabular}



In [67]:

# ============================================================
# 31. FINAL MANIFEST + COMPACT PAPER BUNDLE
# ============================================================

manifest_out = {
    "experiment": "CABLE_PaperReady_Lean10Runs_v1",
    "timestamp": datetime.datetime.now().astimezone().isoformat(),

    "new_training_count": 10,

    "newly_trained_method": "CABLE",

    "reused_baselines": {
        "BUS-BRA": (
            "SWIFT_MHTM_FINAL_OFFICIALCV_CLEAN_AUDIT_V1 "
            "(checkpoint-only clean audit of the frozen best-Dice models)"
        ),
        "BUS-UCLM": (
            "SWIFT_MHTM_BUSUCLM_SUPERVISED_CV_V1 "
            "(completed supervised patient-disjoint five-fold experiment)"
        ),
    },

    "main_architecture_comparison": {
        "selection_rule": "maximum validation foreground Dice for every method",
        "methods": [
            "SwiftMSeg-LGL",
            "MHTM-Local",
            "CABLE-Dice",
        ],
    },

    "cable_boundary_operating_point": {
        "dice_tolerance_absolute": float(
            cfg.dice_tolerance
        ),
        "primary_within_tolerance": "minimum validation HD95",
        "tie_break_1": "maximum validation Boundary-IoU",
        "tie_break_2": "maximum validation Dice",
        "tie_break_3": "earlier epoch",
        "test_set_used_for_selection": False,
    },

    "cable_architecture": {
        "placement": "MiT-B0 Stage 1",
        "local_branch": "MHTM-Local H4 K3",
        "context_resolution": (
            f"{cfg.cable_context_hw}x"
            f"{cfg.cable_context_hw}"
        ),
        "context_heads": int(
            cfg.cable_context_heads
        ),
        "context_scale_init": float(
            cfg.cable_context_scale_init
        ),
        "loss": "cross_entropy",
    },

    "important_interpretation": (
        "CABLE-Boundary is a secondary operating-point analysis. "
        "It is not substituted into the main architecture table against "
        "baselines selected by validation Dice."
    ),
}


with open(
    FINAL_ROOT
    / "EXPERIMENT_MANIFEST.json",
    "w",
) as f:
    json.dump(
        manifest_out,
        f,
        indent=2,
    )


readme = """
CABLE PAPER-READY LEAN 10-RUN EXPERIMENT
========================================

NEW TRAINING
------------
Only CABLE is trained:
- 5 BUS-BRA outer folds
- 5 BUS-UCLM outer folds
Total: 10 trainings.

REUSED BASELINES
----------------
BUS-BRA:
Authoritative clean checkpoint-only audit of the already frozen Swift-LGL
and MHTM best-validation-Dice checkpoints.

BUS-UCLM:
Previously completed supervised patient-disjoint five-fold Swift-LGL and
MHTM experiment.

TWO CABLE CHECKPOINTS FROM EACH TRAINING
----------------------------------------
1. CABLE-Dice:
   maximum validation foreground Dice.

2. CABLE-Boundary:
   among epochs within 0.005 absolute Dice of the maximum, minimum
   validation HD95; tie-break by Boundary-IoU, Dice, then earlier epoch.

PAPER USE
---------
Main architecture table:
Swift-LGL vs MHTM-Local vs CABLE-Dice.

Secondary operating-point analysis:
CABLE-Dice vs CABLE-Boundary.

No test metric is used for checkpoint selection.
""".strip()


with open(
    FINAL_ROOT
    / "README.txt",
    "w",
) as f:
    f.write(
        readme
    )


zip_path = (
    DRIVE_ROOT
    / "CABLE_PaperReady_Lean10Runs_PaperBundle.zip"
)

if zip_path.exists():
    zip_path.unlink()


with zipfile.ZipFile(
    zip_path,
    mode="w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=6,
) as zf:

    for p in sorted(
        FINAL_ROOT.rglob("*")
    ):
        if not p.is_file():
            continue

        # Keep trained checkpoints and large mask arrays on Drive.
        if p.suffix.lower() in {
            ".pt",
            ".npz",
        }:
            continue

        zf.write(
            p,
            arcname=p.relative_to(
                FINAL_ROOT
            ),
        )


local_zip = (
    Path("/content")
    / zip_path.name
    if Path("/content").exists()
    else Path.cwd()
    / zip_path.name
)


if (
    local_zip.resolve()
    != zip_path.resolve()
):
    shutil.copy2(
        zip_path,
        local_zip,
    )


print("\n" + "=" * 94)
print("CABLE LEAN 10-RUN PAPER EXPERIMENT: COMPLETE")
print("=" * 94)
print("New trainings : 10 CABLE runs only")
print("Drive output  :", FINAL_ROOT)
print("Drive ZIP     :", zip_path)
print("Local ZIP     :", local_zip)
print("=" * 94)



CABLE LEAN 10-RUN PAPER EXPERIMENT: COMPLETE
New trainings : 10 CABLE runs only
Drive output  : /content/drive/MyDrive/MHTM_MEDICAL/CABLE_PAPER_READY_LEAN10_V1
Drive ZIP     : /content/drive/MyDrive/MHTM_MEDICAL/CABLE_PaperReady_Lean10Runs_PaperBundle.zip
Local ZIP     : /content/CABLE_PaperReady_Lean10Runs_PaperBundle.zip


In [68]:

# ============================================================
# 32. FINAL PROTOCOL ASSERTIONS
# ============================================================

assert CABLE_VARIANT == "cable_s1"

assert BASELINE_VARIANTS == (
    "swift_lgl",
    "mhtm_hybrid_final",
)

assert cfg.dice_tolerance == 0.005

assert len(
    OUTER_FOLDS
) == 5

print(
    "FINAL PROTOCOL PASS:"
)

print(
    " - baselines reused, not retrained"
)

print(
    " - exactly 10 new CABLE trainings"
)

print(
    " - best-Dice + boundary-aware checkpoints saved per fold"
)

print(
    " - main table uses CABLE-Dice for fair baseline comparison"
)

print(
    " - CABLE-Boundary reported only as secondary operating-point analysis"
)


FINAL PROTOCOL PASS:
 - baselines reused, not retrained
 - exactly 10 new CABLE trainings
 - best-Dice + boundary-aware checkpoints saved per fold
 - main table uses CABLE-Dice for fair baseline comparison
 - CABLE-Boundary reported only as secondary operating-point analysis
